#### 1. 재무데이터 입력하기

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Financial Modeling Prep API - 정리된 데이터 수집 스크립트
분기별 매출 데이터 + 월별 시가총액 데이터 (월말 날짜 통일)
"""

# ==============================================
# 필수 라이브러리 import
# ==============================================
import requests
import pandas as pd
import numpy as np
import calendar

import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
from DATA.stock_invest_function import *

from datetime import datetime, timedelta
from statsmodels.tsa.statespace.sarimax import SARIMAX
from itertools import product

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D


# plotting 설정
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False


In [2]:
# ==============================================
# 설정값들
# ==============================================

# API 키 설정
apikey = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
API_KEY = apikey

# 파라미터 설정
tic_name = 'ANET'
hs_code = '851762'
item_name = 'PSR'
st_date = '2010-01-01'
end_date = '2025-07-31'
today_date = pd.to_datetime(datetime.today().date())

# 설정 변수
USE_EXOGENOUS = True  # True: 외생변수 사용, False: 외생변수 미사용USE_EXOGENOUS = True

# 테스트용 기업
TICKERS = ['SMCI']
ticker = TICKERS[0]
MAX_RETRIES = 2
REQUEST_DELAY = 0.3
RETRY_DELAY = 1.0

# ==============================================
# 유틸리티 함수들
# ==============================================

def test_api_connection():
    """API 연결 테스트"""
    test_url = f"https://financialmodelingprep.com/api/v3/income-statement/AAPL"
    test_params = {'limit': 1, 'apikey': API_KEY, 'period': 'quarter'}

    try:
        response = requests.get(test_url, params=test_params, timeout=10)

        if response.status_code == 401:
            return False, "API 키가 유효하지 않습니다."
        elif response.status_code == 429:
            return False, "API 요청 한도를 초과했습니다."
        elif response.status_code != 200:
            return False, f"API 오류: {response.status_code}"

        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return False, f"API 오류: {data['Error Message']}"
        elif not data:
            return False, "API에서 빈 응답을 받았습니다."

        return True, f"API 연결 성공"

    except Exception as e:
        return False, f"API 연결 실패: {str(e)}"

def convert_to_month_end(date_str):
    """날짜를 해당 월의 월말로 변환"""
    try:
        # 문자열이나 Timestamp를 datetime으로 변환
        if isinstance(date_str, str):
            date_obj = pd.to_datetime(date_str)
        else:
            date_obj = date_str

        # 해당 월의 마지막 날 계산
        year = date_obj.year
        month = date_obj.month
        last_day = calendar.monthrange(year, month)[1]

        # 월말 날짜 생성
        month_end = datetime(year, month, last_day)
        return month_end

    except Exception as e:
        print(f"날짜 변환 오류: {date_str} -> {e}")
        return None

def add_revenue_ttm(df):
    """
    분기별 매출 데이터에 TTM(최근 4분기 합계) 컬럼 추가
    """
    df_copy = df.copy()
    df_copy = df_copy.sort_values(['ticker', 'date'])

    # TTM 계산을 위한 빈 리스트
    ttm_values = []

    # 티커별로 TTM 계산
    for ticker in df_copy['ticker'].unique():
        ticker_data = df_copy[df_copy['ticker'] == ticker].copy()
        ticker_data = ticker_data.sort_values('date')

        # rolling sum으로 최근 4분기 합계 계산
        ticker_data['revenue_ttm'] = ticker_data['revenue'].rolling(window=4, min_periods=1).sum()

        ttm_values.extend(ticker_data['revenue_ttm'].tolist())

    df_copy['revenue_ttm'] = ttm_values
    return df_copy

def fetch_revenue_data(ticker, retry_count=0):
    """분기별 매출 데이터 수집"""
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': API_KEY, 'period': 'quarter'}

    try:
        response = requests.get(url, params=params, timeout=30)

        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"

        data = response.json()

        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"

        if not data:
            return None, "데이터 없음"

        return data, None

    except Exception as e:
        return None, f"오류: {str(e)}"


def check_variables():
    """변수 존재 확인 함수"""
    print("현재 생성된 DataFrame 변수들:")

    vars_to_check = ['revenue_df', 'monthly_df', 'revenue_df_with_ttm']
    for var_name in vars_to_check:
        if var_name in globals():
            df = globals()[var_name]
            if isinstance(df, pd.DataFrame) and not df.empty:
                print(f"✅ {var_name}: {df.shape}")
            else:
                print(f"⚠️ {var_name}: 빈 DataFrame")
        else:
            print(f"❌ {var_name}: 존재하지 않음")



####################################################### SARIMA 예측 함수
# 외생변수를 포함한 SARIMA 예측 함수들
def calculate_available_forecast_periods(quarterly_data, final_data):
    """
    외생변수가 사용 가능한 예측 기간 계산

    Parameters:
    - quarterly_data: 분기별 매출 데이터
    - final_data: expDlr이 포함된 월별 데이터

    Returns:
    - available_periods: 예측 가능한 분기 수
    - last_quarter_date: 마지막 분기 날짜
    - available_forecast_dates: 예측 가능한 날짜들
    """
    # 분기별 데이터의 마지막 날짜
    last_quarterly_date = pd.to_datetime(quarterly_data['date_month_end'].iloc[-1])

    # 외생변수 데이터의 마지막 날짜
    final_data_with_date = final_data.copy()
    final_data_with_date['date'] = pd.to_datetime(final_data_with_date['date_month_end'])
    last_exog_date = final_data_with_date['date'].max()

    print(f"분기별 데이터 마지막 날짜: {last_quarterly_date.strftime('%Y-%m-%d')}")
    print(f"외생변수 데이터 마지막 날짜: {last_exog_date.strftime('%Y-%m-%d')}")

    # 다음 분기들 계산
    available_forecast_dates = []
    current_date = last_quarterly_date

    for i in range(1, 9):  # 최대 8분기까지 체크
        next_quarter_date = current_date + pd.DateOffset(months=3*i)
        quarter_end = pd.Timestamp(
            year=next_quarter_date.year,
            month=next_quarter_date.month,
            day=pd.Timestamp(next_quarter_date.year, next_quarter_date.month, 1).days_in_month
        )

        # 해당 분기말이 외생변수 데이터 범위 내에 있는지 확인
        if quarter_end <= last_exog_date:
            available_forecast_dates.append(quarter_end)
        else:
            break

    available_periods = len(available_forecast_dates)

    print(f"예측 가능한 분기 수: {available_periods}")
    if available_forecast_dates:
        print(f"예측 가능한 날짜: {[d.strftime('%Y-%m') for d in available_forecast_dates]}")
    else:
        print("예측 가능한 분기가 없습니다 (외생변수 데이터 부족)")

    return available_periods, last_quarterly_date, available_forecast_dates

def calculate_yoy_growth_rate(data, value_col='expDlr'):
    """
    YoY (Year over Year) 변화율 계산

    Parameters:
    - data: DataFrame with date and value columns
    - value_col: 값이 있는 컬럼명

    Returns:
    - DataFrame with YoY growth rate column added
    """
    df = data.copy()
    df['date'] = pd.to_datetime(df['date_month_end'])
    df = df.sort_values('date')

    # 연도 추출
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month

    # YoY 변화율 계산 (12개월 전 대비)
    df['yoy_growth_rate'] = df[value_col].pct_change(periods=12) * 100

    print(f"YoY Growth Rate 계산 완료:")
    print(f"  - 데이터 기간: {df['date'].min().strftime('%Y-%m')} ~ {df['date'].max().strftime('%Y-%m')}")
    print(f"  - YoY 값 범위: {df['yoy_growth_rate'].min():.2f}% ~ {df['yoy_growth_rate'].max():.2f}%")
    print(f"  - NaN 개수: {df['yoy_growth_rate'].isna().sum()}")

    return df

def prepare_sarima_data_with_exog(quarterly_data, final_data, USE_EXOGENOUS=True):
    """
    SARIMA를 위한 데이터 준비 (외생변수 포함)

    Parameters:
    - quarterly_data: 분기별 매출 데이터
    - final_data: expDlr이 포함된 월별 데이터
    - USE_EXOGENOUS: 외생변수 사용 여부

    Returns:
    - prepared_data: 준비된 데이터 DataFrame
    """
    # 1. Final_data에서 YoY 변화율 계산
    final_with_yoy = calculate_yoy_growth_rate(final_data, 'expDlr')

    # 2. 분기별 데이터 준비
    quarterly_df = quarterly_data.copy()
    quarterly_df['date'] = pd.to_datetime(quarterly_df['date_month_end'])
    quarterly_df['year'] = quarterly_df['date'].dt.year
    quarterly_df['quarter'] = quarterly_df['date'].dt.quarter

    # 3. 분기말 월 매핑 (Q1=3월, Q2=6월, Q3=9월, Q4=12월)
    quarter_month_map = {1: 3, 2: 6, 3: 9, 4: 12}
    quarterly_df['quarter_end_month'] = quarterly_df['quarter'].map(quarter_month_map)

    # 4. 외생변수 사용 여부에 따른 데이터 준비
    if USE_EXOGENOUS:
        print("외생변수(expDlr YoY) 사용 모드")

        # 분기별 데이터와 월별 외생변수 매칭
        merged_data = []

        for _, row in quarterly_df.iterrows():
            year = row['year']
            month = row['quarter_end_month']

            # 해당 분기말 월의 YoY 변화율 찾기
            matching_exog = final_with_yoy[
                (final_with_yoy['year'] == year) &
                (final_with_yoy['month'] == month)
            ]['yoy_growth_rate']

            if not matching_exog.empty:
                exog_value = matching_exog.iloc[0]
            else:
                # 매칭되는 데이터가 없으면 가장 가까운 값 사용
                closest_date = final_with_yoy[final_with_yoy['date'] <= row['date']]
                if not closest_date.empty:
                    exog_value = closest_date.iloc[-1]['yoy_growth_rate']
                else:
                    exog_value = np.nan

            merged_data.append({
                'date': row['date'],
                'endog_var': row['revenue_billions'],
                'exog_var': exog_value,
                'year': year,
                'quarter': row['quarter']
            })

        prepared_data = pd.DataFrame(merged_data)

        # NaN 처리
        if prepared_data['exog_var'].isna().any():
            print(f"Warning: {prepared_data['exog_var'].isna().sum()} NaN values in exog_var")
            # forward fill 후 backward fill
            prepared_data['exog_var'] = prepared_data['exog_var'].fillna(method='ffill').fillna(method='bfill')
            # 여전히 NaN이 있다면 평균값으로 대체
            if prepared_data['exog_var'].isna().any():
                mean_exog = prepared_data['exog_var'].mean()
                prepared_data['exog_var'] = prepared_data['exog_var'].fillna(mean_exog)

        print(f"외생변수 범위: {prepared_data['exog_var'].min():.2f}% ~ {prepared_data['exog_var'].max():.2f}%")

    else:
        print("외생변수 미사용 모드")
        prepared_data = pd.DataFrame({
            'date': quarterly_df['date'],
            'endog_var': quarterly_df['revenue_billions'],
            'exog_var': None,
            'year': quarterly_df['year'],
            'quarter': quarterly_df['quarter']
        })

    print(f"SARIMA 데이터 준비 완료: {len(prepared_data)}개 분기 데이터")

    return prepared_data

def enhanced_sarima_forecast_with_exog(data, USE_EXOGENOUS=True, forecast_steps=4, use_log=False):
    """
    외생변수를 포함한 향상된 SARIMA 예측

    Parameters:
    - data: 준비된 데이터 (endog_var, exog_var 컬럼 포함)
    - USE_EXOGENOUS: 외생변수 사용 여부
    - forecast_steps: 예측할 스텝 수
    - use_log: 로그 변환 사용 여부

    Returns:
    - forecast_series: 예측 결과
    - param_string: 모델 파라미터 문자열
    - model_info: 모델 정보
    """
    if not USE_EXOGENOUS:
        print("외생변수를 사용하지 않음 - None 반환")
        return None, None, None

    df = data.copy().sort_values('date')
    endog = df['endog_var']

    # 외생변수 준비
    if USE_EXOGENOUS and 'exog_var' in df.columns and df['exog_var'].iloc[0] is not None:
        exog = df['exog_var']
        print(f"외생변수 사용: expDlr YoY 변화율")
        print(f"외생변수 범위: {exog.min():.2f}% ~ {exog.max():.2f}%")
    else:
        exog = None
        print("외생변수 데이터 없음")

    # 로그 변환
    if use_log:
        endog = np.log(endog)
        print("로그 변환 적용")

    # SARIMA 파라미터 그리드
    p_values = [0, 1, 2]
    d_values = [0, 1]
    q_values = [0, 1, 2]
    P_values = [0, 1]
    D_values = [0, 1]
    Q_values = [0, 1]
    s_value = 4  # 분기별 계절성

    best_aic = float('inf')
    best_params = None
    best_model = None

    print("SARIMA 모델 최적화 시작...")

    total_combinations = len(p_values) * len(d_values) * len(q_values) * len(P_values) * len(D_values) * len(Q_values)
    tested_combinations = 0

    for p, d, q, P, D, Q in product(p_values, d_values, q_values, P_values, D_values, Q_values):
        try:
            tested_combinations += 1

            # 파라미터 수 제한 (과적합 방지)
            total_params = p + q + P + Q + 1
            if total_params >= len(endog) * 0.3:
                continue

            # SARIMAX 모델 생성
            model = SARIMAX(
                endog,
                exog=exog,
                order=(p, d, q),
                seasonal_order=(P, D, Q, s_value),
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted_model = model.fit(disp=False, maxiter=100)

            if fitted_model.aic < best_aic:
                best_aic = fitted_model.aic
                best_params = (p, d, q, P, D, Q, s_value)
                best_model = fitted_model

        except Exception as e:
            continue

    # 최적 모델을 찾지 못한 경우 기본 모델 사용
    if best_model is None:
        print("Warning: 최적 모델을 찾지 못해 기본 모델 사용")
        try:
            model = SARIMAX(
                endog,
                exog=exog,
                order=(1, 1, 1),
                seasonal_order=(0, 0, 0, 0),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            best_model = model.fit(disp=False)
            best_params = (1, 1, 1, 0, 0, 0, 0)
        except Exception as e:
            raise ValueError(f"모든 SARIMA 설정이 실패했습니다: {e}")

    print(f"최적 모델 선택 완료: {tested_combinations}/{total_combinations} 조합 테스트")
    print(f"최적 파라미터: {best_params}")
    print(f"최적 AIC: {best_aic:.2f}")

    # 미래 예측
    if exog is not None:
        # 외생변수의 미래 값 생성 (마지막 값 사용 또는 트렌드 연장)
        last_exog_value = exog.iloc[-1]

        # 최근 트렌드 계산 (선택사항)
        recent_trend = 0
        if len(exog) >= 4:
            recent_values = exog.tail(4)
            recent_trend = np.mean(np.diff(recent_values))

        # 미래 외생변수 값 생성
        future_exog = []
        for i in range(forecast_steps):
            # 옵션 1: 마지막 값 유지
            future_value = last_exog_value

            # 옵션 2: 트렌드 연장 (주석 해제하여 사용)
            # future_value = last_exog_value + recent_trend * (i + 1)

            future_exog.append(future_value)

        print(f"미래 외생변수 값: {future_exog}")
        forecast = best_model.forecast(steps=forecast_steps, exog=future_exog)
    else:
        forecast = best_model.forecast(steps=forecast_steps)

    # 로그 변환 역변환
    if use_log:
        forecast = np.exp(forecast)

    # 파라미터 문자열 생성
    param_string = f"({best_params[0]},{best_params[1]},{best_params[2]})({best_params[3]},{best_params[4]},{best_params[5]},{best_params[6]})"

    # 모델 정보
    model_info = {
        'aic': best_aic,
        'params': best_params,
        'exog_used': exog is not None,
        'log_transformed': use_log,
        'future_exog': future_exog if exog is not None else None
    }

    print(f"예측 완료: {forecast.values}")

    return pd.Series(forecast.values), param_string, model_info

def sarima_forecast_comparison(quarterly_data, final_data, USE_EXOGENOUS=True):
    """
    외생변수 사용/미사용 SARIMA 예측 비교 (사용 가능한 기간까지만)

    Parameters:
    - quarterly_data: 분기별 매출 데이터
    - final_data: expDlr이 포함된 월별 데이터
    - USE_EXOGENOUS: 외생변수 사용 여부

    Returns:
    - results: 예측 결과 딕셔너리
    """
    print(f"=== SARIMA 예측 (외생변수 사용: {USE_EXOGENOUS}) ===")

    if not USE_EXOGENOUS:
        print("외생변수를 사용하지 않음 - None 반환")
        return {
            'forecast_values': None,
            'param_string': None,
            'model_info': None,
            'use_exogenous': False,
            'prepared_data': None,
            'available_periods': 0,
            'available_dates': []
        }

    # 1. 예측 가능한 기간 확인
    available_periods, last_quarterly_date, available_forecast_dates = calculate_available_forecast_periods(
        quarterly_data, final_data
    )

    if available_periods == 0:
        print("예측 가능한 기간이 없습니다.")
        return {
            'forecast_values': None,
            'param_string': None,
            'model_info': None,
            'use_exogenous': USE_EXOGENOUS,
            'prepared_data': None,
            'available_periods': 0,
            'available_dates': []
        }

    # 2. 데이터 준비 (예측 기간 포함)
    prepared_data = prepare_sarima_data_with_exog_for_forecast(
        quarterly_data, final_data, available_forecast_dates, USE_EXOGENOUS
    )

    # 3. 예측 수행
    forecast_series, param_string, model_info = enhanced_sarima_forecast_with_exog(
        prepared_data,
        USE_EXOGENOUS=USE_EXOGENOUS,
        max_forecast_steps=available_periods,
        use_log=False
    )

    # 결과 정리
    results = {
        'forecast_values': forecast_series.values if forecast_series is not None else None,
        'param_string': param_string,
        'model_info': model_info,
        'use_exogenous': USE_EXOGENOUS,
        'prepared_data': prepared_data,
        'available_periods': available_periods,
        'available_dates': available_forecast_dates
    }

    return results

def prepare_sarima_data_with_exog_for_forecast(quarterly_data, final_data, forecast_dates, USE_EXOGENOUS=True):
    """
    예측 기간을 포함한 SARIMA 데이터 준비 (expDlr 예측값 사용)

    Parameters:
    - quarterly_data: 분기별 매출 데이터
    - final_data: expDlr이 포함된 월별 데이터 (예측값 포함)
    - forecast_dates: 예측할 날짜들
    - USE_EXOGENOUS: 외생변수 사용 여부

    Returns:
    - prepared_data: 예측 기간을 포함한 준비된 데이터
    """
    if not USE_EXOGENOUS:
        return None

    # 1. Final_data에서 YoY 변화율 계산 (예측값 포함)
    final_with_yoy = calculate_yoy_growth_rate(final_data, 'expDlr')

    # 2. 기존 분기별 데이터 준비
    quarterly_df = quarterly_data.copy()
    quarterly_df['date'] = pd.to_datetime(quarterly_df['date_month_end'])
    quarterly_df['year'] = quarterly_df['date'].dt.year
    quarterly_df['quarter'] = quarterly_df['date'].dt.quarter

    # 3. 분기말 월 매핑
    quarter_month_map = {1: 3, 2: 6, 3: 9, 4: 12}
    quarterly_df['quarter_end_month'] = quarterly_df['quarter'].map(quarter_month_map)

    # 4. 기존 데이터에 외생변수 매칭 (과거 데이터)
    historical_data = []

    for _, row in quarterly_df.iterrows():
        year = row['year']
        month = row['quarter_end_month']

        # 해당 분기말 월의 YoY 변화율 찾기
        matching_exog = final_with_yoy[
            (final_with_yoy['year'] == year) &
            (final_with_yoy['month'] == month)
        ]['yoy_growth_rate']

        if not matching_exog.empty:
            exog_value = matching_exog.iloc[0]
        else:
            # 매칭되는 데이터가 없으면 가장 가까운 값 사용
            closest_date = final_with_yoy[final_with_yoy['date'] <= row['date']]
            if not closest_date.empty:
                exog_value = closest_date.iloc[-1]['yoy_growth_rate']
            else:
                exog_value = np.nan

        historical_data.append({
            'date': row['date'],
            'endog_var': row['revenue_billions'],
            'exog_var': exog_value,
            'year': year,
            'quarter': row['quarter'],
            'is_forecast': False
        })

    # 5. 예측 기간의 외생변수 추가 (expDlr 예측값 사용)
    forecast_data = []

    for forecast_date in forecast_dates:
        forecast_year = forecast_date.year
        forecast_quarter = forecast_date.quarter
        forecast_month = quarter_month_map[forecast_quarter]

        # 예측 기간의 외생변수 값 찾기 (expDlr 예측값이 있는 구간)
        matching_exog = final_with_yoy[
            (final_with_yoy['year'] == forecast_year) &
            (final_with_yoy['month'] == forecast_month)
        ]['yoy_growth_rate']

        if not matching_exog.empty:
            exog_value = matching_exog.iloc[0]
            print(f"예측 기간 {forecast_date.strftime('%Y-Q%m')} 외생변수(YoY): {exog_value:.2f}%")
        else:
            # 정확한 분기말 날짜가 없다면 해당 월에서 가장 가까운 값 찾기
            month_data = final_with_yoy[
                (final_with_yoy['year'] == forecast_year) &
                (final_with_yoy['month'] == forecast_month)
            ]

            if not month_data.empty:
                exog_value = month_data['yoy_growth_rate'].iloc[-1]  # 해당 월 마지막 값
                print(f"예측 기간 {forecast_date.strftime('%Y-Q%m')} 외생변수(YoY): {exog_value:.2f}% (해당 월 값 사용)")
            else:
                # 해당 월 데이터도 없다면 가장 가까운 미래 값 사용
                future_data = final_with_yoy[final_with_yoy['date'] >= forecast_date]
                if not future_data.empty:
                    exog_value = future_data['yoy_growth_rate'].iloc[0]
                    print(f"예측 기간 {forecast_date.strftime('%Y-Q%m')} 외생변수(YoY): {exog_value:.2f}% (가장 가까운 미래 값)")
                else:
                    # 마지막 사용 가능한 값 사용
                    exog_value = final_with_yoy['yoy_growth_rate'].dropna().iloc[-1]
                    print(f"예측 기간 {forecast_date.strftime('%Y-Q%m')} 외생변수(YoY): {exog_value:.2f}% (마지막 값 사용)")

        forecast_data.append({
            'date': forecast_date,
            'endog_var': np.nan,  # 예측할 값
            'exog_var': exog_value,
            'year': forecast_year,
            'quarter': forecast_quarter,
            'is_forecast': True
        })

    # 6. 전체 데이터 결합
    all_data = historical_data + forecast_data
    prepared_data = pd.DataFrame(all_data)

    # 7. NaN 처리 (외생변수만)
    if prepared_data['exog_var'].isna().any():
        print(f"Warning: {prepared_data['exog_var'].isna().sum()} NaN values in exog_var")
        prepared_data['exog_var'] = prepared_data['exog_var'].fillna(method='ffill').fillna(method='bfill')

        if prepared_data['exog_var'].isna().any():
            mean_exog = prepared_data['exog_var'].mean()
            prepared_data['exog_var'] = prepared_data['exog_var'].fillna(mean_exog)

    print(f"준비된 데이터: 과거 {len(historical_data)}분기 + 예측 {len(forecast_data)}분기")
    print(f"외생변수 범위: {prepared_data['exog_var'].min():.2f}% ~ {prepared_data['exog_var'].max():.2f}%")

    return prepared_data
    """
    외생변수 사용/미사용 SARIMA 예측 비교

    Parameters:
    - quarterly_data: 분기별 매출 데이터
    - final_data: expDlr이 포함된 월별 데이터
    - USE_EXOGENOUS: 외생변수 사용 여부
    - forecast_steps: 예측할 스텝 수

    Returns:
    - results: 예측 결과 딕셔너리
    """
    print(f"=== SARIMA 예측 (외생변수 사용: {USE_EXOGENOUS}) ===")

    # 데이터 준비
    prepared_data = prepare_sarima_data_with_exog(quarterly_data, final_data, USE_EXOGENOUS)

    # 예측 수행
    forecast_series, param_string, model_info = enhanced_sarima_forecast_with_exog(
        prepared_data,
        USE_EXOGENOUS=USE_EXOGENOUS,
        forecast_steps=forecast_steps,
        use_log=False
    )

    # 결과 정리
    results = {
        'forecast_values': forecast_series.values if forecast_series is not None else None,
        'param_string': param_string,
        'model_info': model_info,
        'use_exogenous': USE_EXOGENOUS,
        'prepared_data': prepared_data
    }

    return results

def generate_forecast_dates(last_date, periods=4, freq='Q'):
    """예측을 위한 미래 날짜 생성"""
    last_date = pd.to_datetime(last_date)
    future_dates = []

    for i in range(1, periods + 1):
        if freq == 'Q':
            next_quarter_date = last_date + pd.DateOffset(months=3*i)
            quarter_end = pd.Timestamp(
                year=next_quarter_date.year,
                month=next_quarter_date.month,
                day=pd.Timestamp(next_quarter_date.year, next_quarter_date.month, 1).days_in_month
            )
            future_dates.append(quarter_end)

    return future_dates

def create_sarima_exog_result_dataframe(forecast_dates, forecast_values, param_string, use_exogenous):
    """SARIMA 외생변수 예측 결과를 DataFrame으로 생성"""
    if forecast_values is None:
        # 외생변수 미사용 시 None 반환
        return None

    result_df = pd.DataFrame({
        'date': forecast_dates,
        'forecast_revenue_billions': forecast_values,
        'year': [d.year for d in forecast_dates],
        'quarter': [d.quarter for d in forecast_dates],
        'year_quarter': [f"{d.year}Q{d.quarter}" for d in forecast_dates],
        'model_type': f'SARIMA_exog_{param_string}' if use_exogenous else f'SARIMA_{param_string}',
        'exogenous_used': use_exogenous
    })

    return result_df

# 기존 데이터 사용 (SARIMA 코드에서 생성된 quarterly_data 활용)
# quarterly_data가 이미 존재한다고 가정
#@@@@#######################################################################################################



def create_lstm_sequences(data, lookback_window=8):
    """시계열 데이터를 LSTM 입력 형태로 변환"""
    X, y = [], []
    for i in range(lookback_window, len(data)):
        X.append(data[i-lookback_window:i])
        y.append(data[i])
    return np.array(X), np.array(y)

def lstm_forecast(data, lookback_window=8, forecast_steps=4, epochs=100, batch_size=1):
    """
    LSTM을 사용한 시계열 예측 (NaN 처리 포함)
    """
    # 1. 입력 데이터 검증 및 NaN 처리
    if isinstance(data, pd.Series):
        data_values = data.values.astype(np.float64)
    else:
        data_values = np.array(data, dtype=np.float64)

    # NaN 값 확인 및 처리
    if np.isnan(data_values).any():
        print(f"Warning: {np.isnan(data_values).sum()} NaN values found in input data")
        # forward fill로 NaN 처리
        mask = np.isnan(data_values)
        indices = np.where(~mask)[0]
        data_values = np.interp(np.arange(len(data_values)), indices, data_values[indices])
        print("NaN values filled using interpolation")

    print(f"Data range: {data_values.min():.2f} to {data_values.max():.2f}")
    print(f"Data length: {len(data_values)}")

    # 2. 데이터 정규화
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data_values.reshape(-1, 1))

    # 정규화 후 NaN 체크
    if np.isnan(scaled_data).any():
        print("Warning: NaN values found after scaling")
        return None, None, None

    # 3. 시퀀스 데이터 생성
    def create_sequences(data, lookback_window):
        X, y = [], []
        for i in range(lookback_window, len(data)):
            X.append(data[i-lookback_window:i, 0])
            y.append(data[i, 0])
        return np.array(X), np.array(y)

    if len(scaled_data) <= lookback_window:
        print(f"Error: Not enough data points. Need at least {lookback_window + 1} points.")
        return None, None, None

    X, y = create_sequences(scaled_data, lookback_window)

    # 시퀀스 생성 후 NaN 체크
    if np.isnan(X).any() or np.isnan(y).any():
        print("Warning: NaN values found in sequences")
        return None, None, None

    # 4. LSTM 입력을 위한 reshape
    X = X.reshape((X.shape[0], X.shape[1], 1))

    print(f"Training data shape: X={X.shape}, y={y.shape}")

    # 5. LSTM 모델 구성
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=(lookback_window, 1)))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dense(25))
    model.add(Dense(1))

    # 6. 모델 컴파일 및 훈련
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    try:
        history = model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=0)
        print(f"Training completed. Final loss: {history.history['loss'][-1]:.6f}")
    except Exception as e:
        print(f"Training failed: {e}")
        return None, None, None

    # 7. 미래 예측
    last_sequence = scaled_data[-lookback_window:].reshape(1, lookback_window, 1)

    predictions = []
    current_sequence = last_sequence.copy()

    for _ in range(forecast_steps):
        # 예측 수행
        pred = model.predict(current_sequence, verbose=0)

        # NaN 체크
        if np.isnan(pred).any():
            print("Warning: NaN prediction detected")
            # 이전 값의 평균으로 대체
            if predictions:
                pred = np.array([[np.mean(predictions)]])
            else:
                pred = np.array([[current_sequence[0, -1, 0]]])

        predictions.append(pred[0, 0])

        # 다음 시퀀스를 위해 업데이트
        current_sequence = np.roll(current_sequence, -1, axis=1)
        current_sequence[0, -1, 0] = pred[0, 0]

    # 8. 역정규화
    predictions_array = np.array(predictions).reshape(-1, 1)

    # 역정규화 전 NaN 체크
    if np.isnan(predictions_array).any():
        print("Warning: NaN in predictions before inverse transform")
        predictions_array = np.nan_to_num(predictions_array, nan=np.mean(scaled_data))

    try:
        forecast_values = scaler.inverse_transform(predictions_array).flatten()

        # 최종 결과 NaN 체크
        if np.isnan(forecast_values).any():
            print("Warning: NaN in final forecast values")
            # 마지막 실제값으로 대체
            last_actual = data_values[-1]
            forecast_values = np.nan_to_num(forecast_values, nan=last_actual)

        print(f"Forecast completed: {forecast_values}")
        return forecast_values, model, scaler

    except Exception as e:
        print(f"Inverse transform failed: {e}")
        return None, None, None

def evaluate_model_performance(actual_data, model, scaler, lookback_window=8):
    """
    LSTM 모델 성능 평가 (NaN 처리 강화)
    """
    try:
        # 1. 입력 데이터 처리
        if isinstance(actual_data, pd.Series):
            data_values = actual_data.values.astype(np.float64)
        else:
            data_values = np.array(actual_data, dtype=np.float64)

        # NaN 처리
        if np.isnan(data_values).any():
            print("NaN found in actual data, filling with interpolation")
            mask = np.isnan(data_values)
            indices = np.where(~mask)[0]
            if len(indices) == 0:
                return {'MSE': np.nan, 'MAE': np.nan, 'RMSE': np.nan, 'Error': 'All data is NaN'}
            data_values = np.interp(np.arange(len(data_values)), indices, data_values[indices])

        # 2. 충분한 데이터가 있는지 확인
        if len(data_values) <= lookback_window + 1:
            return {'MSE': np.nan, 'MAE': np.nan, 'RMSE': np.nan,
                    'Error': f'Not enough data for evaluation. Need > {lookback_window + 1} points'}

        # 3. 데이터 분할 (마지막 20% 또는 최소 4개를 테스트용으로)
        test_size = max(4, len(data_values) // 5)
        train_data = data_values[:-test_size]
        test_data = data_values[-test_size:]

        print(f"Evaluation: train_size={len(train_data)}, test_size={len(test_data)}")

        # 4. 정규화
        scaled_data = scaler.transform(train_data.reshape(-1, 1))

        # 5. 시퀀스 생성 함수
        def create_sequences(data, lookback):
            X, y = [], []
            for i in range(lookback, len(data)):
                X.append(data[i-lookback:i, 0])
                y.append(data[i, 0])
            return np.array(X), np.array(y)

        # 6. 예측 수행
        predictions = []
        current_data = scaled_data.copy()

        for i in range(test_size):
            if len(current_data) >= lookback_window:
                # 시퀀스 생성
                last_sequence = current_data[-lookback_window:].reshape(1, lookback_window, 1)

                # 예측
                pred = model.predict(last_sequence, verbose=0)

                # NaN 체크 및 처리
                if np.isnan(pred).any():
                    if predictions:
                        pred = np.array([[np.mean(predictions)]])
                    else:
                        pred = np.array([[current_data[-1, 0]]])

                predictions.append(pred[0, 0])

                # 실제값을 추가하여 다음 예측을 위한 시퀀스 업데이트
                actual_scaled = scaler.transform([[test_data[i]]])
                current_data = np.vstack([current_data, actual_scaled])
            else:
                # 시퀀스가 부족한 경우 마지막 값으로 예측
                predictions.append(current_data[-1, 0] if len(current_data) > 0 else 0)

        # 7. 역정규화
        if len(predictions) == 0:
            return {'MSE': np.nan, 'MAE': np.nan, 'RMSE': np.nan, 'Error': 'No predictions generated'}

        predictions_array = np.array(predictions).reshape(-1, 1)

        # NaN 체크
        if np.isnan(predictions_array).any():
            print("NaN in predictions, replacing with mean")
            predictions_array = np.nan_to_num(predictions_array, nan=np.nanmean(data_values))

        try:
            pred_values = scaler.inverse_transform(predictions_array).flatten()

            # 최종 NaN 체크
            if np.isnan(pred_values).any():
                pred_values = np.nan_to_num(pred_values, nan=np.nanmean(test_data))

            # 길이 맞추기
            min_length = min(len(test_data), len(pred_values))
            test_data_final = test_data[:min_length]
            pred_values_final = pred_values[:min_length]

            # 8. 성능 지표 계산
            if len(test_data_final) == 0 or len(pred_values_final) == 0:
                return {'MSE': np.nan, 'MAE': np.nan, 'RMSE': np.nan, 'Error': 'Empty predictions'}

            mse = mean_squared_error(test_data_final, pred_values_final)
            mae = mean_absolute_error(test_data_final, pred_values_final)
            rmse = np.sqrt(mse)

            print(f"Performance - MSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}")

            return {
                'MSE': mse,
                'MAE': mae,
                'RMSE': rmse,
                'test_size': len(test_data_final),
                'predictions': pred_values_final.tolist(),
                'actual': test_data_final.tolist()
            }

        except Exception as e:
            print(f"Inverse transform or metric calculation failed: {e}")
            return {'MSE': np.nan, 'MAE': np.nan, 'RMSE': np.nan, 'Error': str(e)}

    except Exception as e:
        print(f"Model evaluation failed: {e}")
        return {'MSE': np.nan, 'MAE': np.nan, 'RMSE': np.nan, 'Error': str(e)}

# 데이터 검증 함수
def check_data_quality(data, name="data"):
    """데이터 품질 확인"""
    if isinstance(data, pd.Series):
        values = data.values
    else:
        values = np.array(data)

    print(f"\n=== {name} Quality Check ===")
    print(f"Length: {len(values)}")
    print(f"NaN count: {np.isnan(values).sum()}")
    print(f"Infinite count: {np.isinf(values).sum()}")
    print(f"Min value: {np.nanmin(values):.4f}")
    print(f"Max value: {np.nanmax(values):.4f}")
    print(f"Mean value: {np.nanmean(values):.4f}")
    print(f"Data type: {values.dtype}")

    return {
        'length': len(values),
        'nan_count': np.isnan(values).sum(),
        'inf_count': np.isinf(values).sum(),
        'min_val': np.nanmin(values),
        'max_val': np.nanmax(values),
        'mean_val': np.nanmean(values)
    }

def check_data_quality(data, name="data"):
    """데이터 품질 확인 함수"""
    if isinstance(data, pd.Series):
        values = data.values
    else:
        values = np.array(data)

    print(f"\n=== {name} Quality Check ===")
    print(f"Length: {len(values)}")
    print(f"NaN count: {np.isnan(values).sum()}")
    print(f"Infinite count: {np.isinf(values).sum()}")
    print(f"Min value: {np.nanmin(values):.4f}")
    print(f"Max value: {np.nanmax(values):.4f}")
    print(f"Mean value: {np.nanmean(values):.4f}")
    print(f"Data type: {values.dtype}")

    return {
        'length': len(values),
        'nan_count': np.isnan(values).sum(),
        'inf_count': np.isinf(values).sum(),
        'min_val': np.nanmin(values),
        'max_val': np.nanmax(values),
        'mean_val': np.nanmean(values)
    }

def prepare_prophet_data(dates, values, date_column='ds', value_column='y'):
    """
    Prophet을 위한 데이터 준비

    Parameters:
    - dates: 날짜 데이터 (Series 또는 list)
    - values: 값 데이터 (Series 또는 list)
    - date_column: Prophet에서 사용할 날짜 컬럼명 (기본: 'ds')
    - value_column: Prophet에서 사용할 값 컬럼명 (기본: 'y')

    Returns:
    - DataFrame: Prophet 형식의 데이터프레임
    """
    # 데이터 변환
    if isinstance(dates, pd.Series):
        dates_list = dates.tolist()
    else:
        dates_list = list(dates)

    if isinstance(values, pd.Series):
        values_list = values.tolist()
    else:
        values_list = list(values)

    # 날짜 변환
    dates_converted = []
    for date in dates_list:
        if isinstance(date, str):
            dates_converted.append(pd.to_datetime(date))
        else:
            dates_converted.append(pd.to_datetime(date))

    # DataFrame 생성
    df = pd.DataFrame({
        date_column: dates_converted,
        value_column: values_list
    })

    # NaN 처리
    if df[value_column].isna().any():
        print(f"Warning: {df[value_column].isna().sum()} NaN values found in values")
        # forward fill 후 backward fill
        df[value_column] = df[value_column].fillna(method='ffill').fillna(method='bfill')
        print("NaN values filled using forward/backward fill")

    # 정렬
    df = df.sort_values(date_column).reset_index(drop=True)

    print(f"Prophet data prepared: {len(df)} records from {df[date_column].min()} to {df[date_column].max()}")

    return df

def prophet_forecast(df, forecast_periods=4, seasonality_mode='additive',
                    yearly_seasonality=True, quarterly_seasonality=True,
                    growth='linear', changepoint_prior_scale=0.05):
    """
    Prophet을 사용한 시계열 예측

    Parameters:
    - df: Prophet 형식 데이터프레임 (ds, y 컬럼 필요)
    - forecast_periods: 예측할 기간 수 (기본: 4)
    - seasonality_mode: 계절성 모드 ('additive' 또는 'multiplicative')
    - yearly_seasonality: 연간 계절성 사용 여부
    - quarterly_seasonality: 분기 계절성 사용 여부
    - growth: 성장 트렌드 ('linear' 또는 'logistic')
    - changepoint_prior_scale: 트렌드 변화점 민감도

    Returns:
    - forecast_df: 예측 결과 DataFrame
    - model: 훈련된 Prophet 모델
    - future_df: 미래 날짜 DataFrame
    - full_forecast: 전체 예측 결과
    """
    try:
        # Prophet 모델 생성
        model = Prophet(
            growth=growth,
            seasonality_mode=seasonality_mode,
            yearly_seasonality=yearly_seasonality,
            weekly_seasonality=False,  # 주간 계절성은 분기 데이터에서 불필요
            daily_seasonality=False,   # 일간 계절성도 불필요
            changepoint_prior_scale=changepoint_prior_scale
        )

        # 분기 계절성 추가 (필요한 경우)
        if quarterly_seasonality:
            model.add_seasonality(name='quarterly', period=365.25/4, fourier_order=4)

        print("Fitting Prophet model...")

        # 모델 훈련
        model.fit(df)

        # 미래 날짜 생성
        future_df = model.make_future_dataframe(periods=forecast_periods, freq='QS')

        print(f"Making predictions for {forecast_periods} periods...")

        # 예측 수행
        forecast = model.predict(future_df)

        # 예측 결과에서 미래 부분만 추출
        forecast_only = forecast.tail(forecast_periods).copy()

        # NaN 체크
        if forecast_only['yhat'].isna().any():
            print("Warning: NaN values in forecast, filling with trend values")
            forecast_only['yhat'] = forecast_only['yhat'].fillna(forecast_only['trend'])

        print(f"Forecast completed. Values range: {forecast_only['yhat'].min():.2f} to {forecast_only['yhat'].max():.2f}")

        return forecast_only, model, future_df, forecast

    except Exception as e:
        print(f"Prophet forecast failed: {e}")
        return None, None, None, None

def evaluate_prophet_performance(df, model, test_size=4):
    """
    Prophet 모델 성능 평가

    Parameters:
    - df: 원본 데이터
    - model: 훈련된 Prophet 모델
    - test_size: 테스트 데이터 크기

    Returns:
    - performance_dict: 성능 지표 딕셔너리
    """
    try:
        if len(df) <= test_size:
            return {'Error': f'Not enough data for evaluation. Need > {test_size} points'}

        # 데이터 분할
        train_data = df.iloc[:-test_size].copy()
        test_data = df.iloc[-test_size:].copy()

        print(f"Performance evaluation: train_size={len(train_data)}, test_size={len(test_data)}")

        # 새 모델로 훈련 (테스트 데이터 제외)
        eval_model = Prophet(
            growth=model.growth,
            seasonality_mode=model.seasonality_mode,
            yearly_seasonality=model.yearly_seasonality,
            weekly_seasonality=model.weekly_seasonality,
            daily_seasonality=model.daily_seasonality,
            changepoint_prior_scale=model.changepoint_prior_scale
        )

        # 분기 계절성이 있다면 추가
        if hasattr(model, 'seasonalities') and 'quarterly' in [s['name'] for s in model.seasonalities.values()]:
            eval_model.add_seasonality(name='quarterly', period=365.25/4, fourier_order=4)

        eval_model.fit(train_data)

        # 테스트 기간 예측
        future = eval_model.make_future_dataframe(periods=test_size, freq='QS')
        forecast = eval_model.predict(future)

        # 예측값 추출
        predictions = forecast.tail(test_size)['yhat'].values
        actual_values = test_data['y'].values

        # NaN 처리
        if np.isnan(predictions).any():
            predictions = np.nan_to_num(predictions, nan=np.nanmean(actual_values))

        # 성능 지표 계산
        mse = mean_squared_error(actual_values, predictions)
        mae = mean_absolute_error(actual_values, predictions)
        rmse = np.sqrt(mse)

        # 상대 오차 계산
        mape = np.mean(np.abs((actual_values - predictions) / actual_values)) * 100

        print(f"Performance - MSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.2f}%")

        return {
            'MSE': mse,
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape,
            'test_size': len(actual_values),
            'predictions': predictions.tolist(),
            'actual': actual_values.tolist()
        }

    except Exception as e:
        print(f"Performance evaluation failed: {e}")
        return {'Error': str(e)}

def generate_forecast_dates(last_date, periods=4, freq='Q'):
    """
    예측을 위한 미래 날짜 생성

    Parameters:
    - last_date: 마지막 날짜
    - periods: 생성할 기간 수
    - freq: 빈도 ('Q' for quarterly, 'M' for monthly, 'Y' for yearly)

    Returns:
    - list: 미래 날짜 리스트
    """
    last_date = pd.to_datetime(last_date)
    future_dates = []

    for i in range(1, periods + 1):
        if freq == 'Q':
            # 분기별
            next_date = last_date + pd.DateOffset(months=3*i)
            # 분기 말일로 조정
            quarter_end = pd.Timestamp(
                year=next_date.year,
                month=next_date.month,
                day=pd.Timestamp(next_date.year, next_date.month, 1).days_in_month
            )
            future_dates.append(quarter_end)
        elif freq == 'M':
            # 월별
            next_date = last_date + pd.DateOffset(months=i)
            month_end = pd.Timestamp(
                year=next_date.year,
                month=next_date.month,
                day=pd.Timestamp(next_date.year, next_date.month, 1).days_in_month
            )
            future_dates.append(month_end)
        elif freq == 'Y':
            # 연별
            next_date = last_date + pd.DateOffset(years=i)
            future_dates.append(next_date)

    return future_dates

def create_forecast_result_dataframe(forecast_df, date_column='ds'):
    """
    예측 결과를 정리된 DataFrame으로 생성

    Parameters:
    - forecast_df: Prophet 예측 결과 DataFrame
    - date_column: 날짜 컬럼명

    Returns:
    - DataFrame: 정리된 예측 결과
    """
    result_df = pd.DataFrame()

    result_df['date'] = forecast_df[date_column]
    result_df['forecast_revenue_billions'] = forecast_df['yhat']
    result_df['forecast_lower'] = forecast_df['yhat_lower']
    result_df['forecast_upper'] = forecast_df['yhat_upper']
    result_df['trend'] = forecast_df['trend']

    # 날짜 관련 정보 추가
    result_df['year'] = result_df['date'].dt.year
    result_df['quarter'] = result_df['date'].dt.quarter
    result_df['year_quarter'] = result_df['year'].astype(str) + 'Q' + result_df['quarter'].astype(str)

    return result_df

# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D

# Exponential Smoothing 시계열 예측 함수들
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def check_data_quality(data, name="data"):
    """데이터 품질 확인 함수"""
    if isinstance(data, pd.Series):
        values = data.values
    else:
        values = np.array(data)

    print(f"\n=== {name} Quality Check ===")
    print(f"Length: {len(values)}")
    print(f"NaN count: {np.isnan(values).sum()}")
    print(f"Infinite count: {np.isinf(values).sum()}")
    print(f"Min value: {np.nanmin(values):.4f}")
    print(f"Max value: {np.nanmax(values):.4f}")
    print(f"Mean value: {np.nanmean(values):.4f}")
    print(f"Data type: {values.dtype}")

    return {
        'length': len(values),
        'nan_count': np.isnan(values).sum(),
        'inf_count': np.isinf(values).sum(),
        'min_val': np.nanmin(values),
        'max_val': np.nanmax(values),
        'mean_val': np.nanmean(values)
    }

def prepare_exponential_smoothing_data(data):
    """
    Exponential Smoothing을 위한 데이터 준비

    Parameters:
    - data: 시계열 데이터 (Series 또는 array)

    Returns:
    - clean_data: 정제된 시계열 데이터
    """
    if isinstance(data, pd.Series):
        clean_data = data.copy()
    else:
        clean_data = pd.Series(data)

    # NaN 처리
    if clean_data.isna().any():
        print(f"Warning: {clean_data.isna().sum()} NaN values found")
        # forward fill 후 backward fill
        clean_data = clean_data.fillna(method='ffill').fillna(method='bfill')
        print("NaN values filled using forward/backward fill")

    # 무한값 처리
    if np.isinf(clean_data).any():
        print("Warning: Infinite values found, replacing with median")
        median_val = clean_data.median()
        clean_data = clean_data.replace([np.inf, -np.inf], median_val)

    print(f"Data prepared: {len(clean_data)} records, range: {clean_data.min():.2f} to {clean_data.max():.2f}")

    return clean_data

def safe_param_print(param_name, param_value):
    """파라미터를 안전하게 출력하는 함수"""
    try:
        if param_value is None:
            return f"  - {param_name}: None"
        elif isinstance(param_value, (list, np.ndarray)):
            if len(param_value) == 0:
                return f"  - {param_name}: empty"
            elif len(param_value) == 1:
                return f"  - {param_name}: {float(param_value[0]):.4f}"
            else:
                return f"  - {param_name}: {[float(x) for x in param_value[:3]]}"  # 처음 3개만
        elif np.isnan(float(param_value)):
            return f"  - {param_name}: nan"
        else:
            return f"  - {param_name}: {float(param_value):.4f}"
    except (ValueError, TypeError, IndexError):
        return f"  - {param_name}: {str(param_value)}"

def exponential_smoothing_forecast_fixed(data, forecast_steps=4, damped=False,
                                        seasonal='add', seasonal_periods=4,
                                        trend='add', auto_optimize=True):
    """
    수정된 Exponential Smoothing 예측 함수

    Parameters:
    - data: 시계열 데이터
    - forecast_steps: 예측할 스텝 수 (기본: 4)
    - damped: Damped 트렌드 사용 여부 (기본: False)
    - seasonal: 계절성 유형 ('add', 'mul', None)
    - seasonal_periods: 계절성 주기 (기본: 4 for quarterly)
    - trend: 트렌드 유형 ('add', 'mul', None)
    - auto_optimize: 자동 파라미터 최적화 여부

    Returns:
    - forecast_values: 예측값
    - model: 훈련된 모델
    - forecast_result: 상세 예측 결과
    """
    try:
        print(f"=== Exponential Smoothing 예측 시작 (Damped: {damped}) ===")

        # 데이터 준비
        clean_data = prepare_exponential_smoothing_data(data)

        if len(clean_data) < 2:
            print("Error: 예측을 위해 최소 2개의 데이터 포인트가 필요합니다.")
            return None, None, None

        # 계절성 설정 조정 (데이터 길이에 따라)
        if len(clean_data) < seasonal_periods * 2:
            print(f"Warning: 데이터가 부족하여 계절성을 제거합니다 (필요: {seasonal_periods * 2}, 실제: {len(clean_data)})")
            seasonal = None
            seasonal_periods = None

        print(f"Model configuration:")
        print(f"  - Trend: {trend}")
        print(f"  - Seasonal: {seasonal}")
        print(f"  - Seasonal periods: {seasonal_periods}")
        print(f"  - Damped: {damped}")
        print(f"  - Auto optimize: {auto_optimize}")

        # 모델 생성
        model_params = {
            'trend': trend,
            'seasonal': seasonal,
            'damped_trend': damped,
        }

        # 계절성 주기 설정
        if seasonal is not None and seasonal_periods is not None:
            model_params['seasonal_periods'] = seasonal_periods

        # 모델 생성
        model = ExponentialSmoothing(clean_data, **model_params)

        # 모델 피팅
        print("Fitting Exponential Smoothing model...")

        if auto_optimize:
            # 자동 최적화
            fitted_model = model.fit(optimized=True, use_brute=False)  # use_brute=False로 변경
        else:
            # 기본 파라미터 사용
            fitted_model = model.fit(optimized=False)

        print("Model fitting completed")

        # 모델 파라미터 출력 (안전하게)
        print(f"Fitted parameters:")
        if hasattr(fitted_model, 'params') and fitted_model.params is not None:
            for param_name, param_value in fitted_model.params.items():
                print(safe_param_print(param_name, param_value))

        # 예측 수행
        print(f"Generating {forecast_steps} forecasts...")
        try:
            forecast_result = fitted_model.forecast(steps=forecast_steps)
        except Exception as e:
            print(f"Forecast generation failed: {e}")
            # 대안: 마지막 값의 평균 증가율로 예측
            recent_trend = np.mean(np.diff(clean_data.tail(4)))
            last_value = clean_data.iloc[-1]
            forecast_result = [last_value + recent_trend * (i+1) for i in range(forecast_steps)]
            forecast_result = pd.Series(forecast_result)
            print(f"Using simple trend-based forecast: {forecast_result.values}")

        # 신뢰구간 계산 (가능한 경우)
        forecast_lower = None
        forecast_upper = None
        try:
            prediction_results = fitted_model.get_prediction(
                start=len(clean_data),
                end=len(clean_data) + forecast_steps - 1
            )
            forecast_ci = prediction_results.conf_int()
            forecast_lower = forecast_ci.iloc[:, 0].values
            forecast_upper = forecast_ci.iloc[:, 1].values
        except Exception as e:
            print(f"신뢰구간 계산 실패: {e}")
            # 기본 신뢰구간 생성
            forecast_std = np.std(clean_data) * 0.1
            if isinstance(forecast_result, pd.Series):
                forecast_lower = forecast_result.values - 1.96 * forecast_std
                forecast_upper = forecast_result.values + 1.96 * forecast_std
            else:
                forecast_lower = np.array(forecast_result) - 1.96 * forecast_std
                forecast_upper = np.array(forecast_result) + 1.96 * forecast_std

        # 예측값 검증
        if isinstance(forecast_result, pd.Series):
            forecast_values = forecast_result.values
        else:
            forecast_values = np.array(forecast_result)

        # NaN 체크
        if np.isnan(forecast_values).any():
            print("Warning: NaN values in forecast, using last known value")
            last_value = clean_data.iloc[-1]
            forecast_values = np.nan_to_num(forecast_values, nan=last_value)

        # 무한값 체크
        if np.isinf(forecast_values).any():
            print("Warning: Infinite values in forecast, using last known value")
            last_value = clean_data.iloc[-1]
            forecast_values = np.where(np.isinf(forecast_values), last_value, forecast_values)

        print(f"Forecast completed: {forecast_values}")

        # 상세 결과 생성
        detailed_forecast = {
            'forecast': forecast_values,
            'forecast_lower': forecast_lower,
            'forecast_upper': forecast_upper,
            'fitted_values': fitted_model.fittedvalues.values if hasattr(fitted_model, 'fittedvalues') else None,
            'residuals': fitted_model.resid.values if hasattr(fitted_model, 'resid') else None,
            'aic': getattr(fitted_model, 'aic', None),
            'bic': getattr(fitted_model, 'bic', None),
            'params': fitted_model.params if hasattr(fitted_model, 'params') else None
        }

        return forecast_values, fitted_model, detailed_forecast

    except Exception as e:
        print(f"Exponential Smoothing forecast failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

def evaluate_exponential_smoothing_performance_fixed(data, damped=False, seasonal='add',
                                                    seasonal_periods=4, trend='add', test_size=4):
    """
    수정된 Exponential Smoothing 모델 성능 평가
    """
    try:
        clean_data = prepare_exponential_smoothing_data(data)

        if len(clean_data) <= test_size:
            return {'Error': f'Not enough data for evaluation. Need > {test_size} points'}

        # 데이터 분할
        train_data = clean_data.iloc[:-test_size]
        test_data = clean_data.iloc[-test_size:]

        print(f"Performance evaluation: train_size={len(train_data)}, test_size={len(test_data)}")

        # 계절성 조정
        eval_seasonal = seasonal
        eval_seasonal_periods = seasonal_periods

        if len(train_data) < seasonal_periods * 2:
            eval_seasonal = None
            eval_seasonal_periods = None

        # 모델 훈련
        forecast_values, model, _ = exponential_smoothing_forecast_fixed(
            train_data,
            forecast_steps=test_size,
            damped=damped,
            seasonal=eval_seasonal,
            seasonal_periods=eval_seasonal_periods,
            trend=trend,
            auto_optimize=True
        )

        if forecast_values is None:
            return {'Error': 'Model fitting failed during evaluation'}

        # 성능 지표 계산
        actual_values = test_data.values

        mse = mean_squared_error(actual_values, forecast_values)
        mae = mean_absolute_error(actual_values, forecast_values)
        rmse = np.sqrt(mse)

        # MAPE 계산 (0으로 나누기 방지)
        mape_values = []
        for actual, pred in zip(actual_values, forecast_values):
            if actual != 0:
                mape_values.append(abs((actual - pred) / actual))
        mape = np.mean(mape_values) * 100 if mape_values else np.nan

        print(f"Performance - MSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.2f}%")

        return {
            'MSE': mse,
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape,
            'test_size': len(actual_values),
            'predictions': forecast_values.tolist(),
            'actual': actual_values.tolist(),
            'damped': damped,
            'seasonal': seasonal,
            'trend': trend
        }

    except Exception as e:
        print(f"Performance evaluation failed: {e}")
        return {'Error': str(e)}

def compare_damped_vs_non_damped_fixed(data, forecast_steps=4, seasonal='add',
                                      seasonal_periods=4, trend='add'):
    """
    수정된 Damped vs Non-Damped Exponential Smoothing 비교
    """
    print("=== Damped vs Non-Damped 비교 ===")

    results = {}

    # Non-Damped 예측
    print("\n1. Non-Damped 예측:")
    non_damped_forecast, non_damped_model, non_damped_details = exponential_smoothing_forecast_fixed(
        data, forecast_steps=forecast_steps, damped=False,
        seasonal=seasonal, seasonal_periods=seasonal_periods, trend=trend
    )

    # Damped 예측
    print("\n2. Damped 예측:")
    damped_forecast, damped_model, damped_details = exponential_smoothing_forecast_fixed(
        data, forecast_steps=forecast_steps, damped=True,
        seasonal=seasonal, seasonal_periods=seasonal_periods, trend=trend
    )

    # 성능 평가
    print("\n3. 성능 비교:")
    non_damped_performance = evaluate_exponential_smoothing_performance_fixed(
        data, damped=False, seasonal=seasonal, seasonal_periods=seasonal_periods, trend=trend
    )

    damped_performance = evaluate_exponential_smoothing_performance_fixed(
        data, damped=True, seasonal=seasonal, seasonal_periods=seasonal_periods, trend=trend
    )

    # 결과 정리
    results = {
        'non_damped': {
            'forecast': non_damped_forecast,
            'model': non_damped_model,
            'details': non_damped_details,
            'performance': non_damped_performance
        },
        'damped': {
            'forecast': damped_forecast,
            'model': damped_model,
            'details': damped_details,
            'performance': damped_performance
        }
    }

    # 최적 모델 선택 (MSE 기준)
    if (non_damped_performance.get('MSE') is not None and
        damped_performance.get('MSE') is not None):

        if non_damped_performance['MSE'] < damped_performance['MSE']:
            results['best_model'] = 'non_damped'
        else:
            results['best_model'] = 'damped'

        print(f"\n최적 모델: {results['best_model']}")
        print(f"Non-Damped MSE: {non_damped_performance['MSE']:.4f}")
        print(f"Damped MSE: {damped_performance['MSE']:.4f}")

    return results

def generate_forecast_dates(last_date, periods=4, freq='Q'):
    """예측을 위한 미래 날짜 생성"""
    last_date = pd.to_datetime(last_date)
    future_dates = []

    for i in range(1, periods + 1):
        if freq == 'Q':
            next_quarter_date = last_date + pd.DateOffset(months=3*i)
            quarter_end = pd.Timestamp(
                year=next_quarter_date.year,
                month=next_quarter_date.month,
                day=pd.Timestamp(next_quarter_date.year, next_quarter_date.month, 1).days_in_month
            )
            future_dates.append(quarter_end)

    return future_dates

def create_exponential_smoothing_result_dataframe(forecast_dates, forecast_values,
                                                 forecast_lower=None, forecast_upper=None,
                                                 damped=False):
    """Exponential Smoothing 예측 결과를 DataFrame으로 생성"""
    result_df = pd.DataFrame({
        'date': forecast_dates,
        'forecast_revenue_billions': forecast_values,
        'year': [d.year for d in forecast_dates],
        'quarter': [d.quarter for d in forecast_dates],
        'year_quarter': [f"{d.year}Q{d.quarter}" for d in forecast_dates],
        'model_type': 'ExponentialSmoothing_Damped' if damped else 'ExponentialSmoothing_NonDamped'
    })

    # 신뢰구간 추가 (있는 경우)
    if forecast_lower is not None:
        result_df['forecast_lower'] = forecast_lower
    if forecast_upper is not None:
        result_df['forecast_upper'] = forecast_upper

    return result_df

In [8]:
# ==============================================
# API 키 확인 및 연결 테스트
# ==============================================

# API 키 유효성 검사
if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("❌ 유효한 API 키를 설정해주세요!")
    print("현재 API_KEY:", API_KEY[:10] + "..." if API_KEY else "None")
else:
    print(f"✅ API 키 설정됨: {API_KEY[:10]}...")

print("🔍 API 연결 테스트 중...")
api_ok, api_message = test_api_connection()
print(api_message)

if not api_ok:
    print("⚠️ API 연결에 실패했지만 계속 진행합니다. API 키와 네트워크 상태를 확인해주세요.")

print(f"🚀 경량 데이터 수집 시작")
print(f"📊 대상 기업: {len(TICKERS)}개 - {TICKERS}")
print(f"📈 수집 데이터: 분기별 매출 + 월별 시가총액")
print("=" * 70)

# ==============================================
# 1. 분기별 매출 데이터 수집
# ==============================================
print("📈 Fetching quarterly revenue data...")
all_revenue_data = []
revenue_successful_tickers = []

for ticker in tqdm(TICKERS, desc="Revenue"):
    revenue_data, error = fetch_revenue_data(ticker)

    if revenue_data is None:
        print(f"   ❌ {ticker}: {error}")
        continue

    for item in revenue_data:
        all_revenue_data.append({
            'ticker': ticker,
            'date': item.get('date', ''),
            'calendar_year': item.get('calendarYear', ''),
            'period': item.get('period', ''),
            'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
            'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
            'gross_profit': item.get('grossProfit', 0) if item.get('grossProfit') is not None else 0,
            'gross_margin': round(((item.get('grossProfit', 0) or 0) / (item.get('revenue', 1) or 1)) * 100, 2) if (item.get('revenue') or 0) > 0 else 0,
        })

    revenue_successful_tickers.append(ticker)
    print(f"   ✅ {ticker}: {len([d for d in all_revenue_data if d['ticker'] == ticker])}개 분기")
    time.sleep(REQUEST_DELAY)

# DataFrame 생성
revenue_df = pd.DataFrame(all_revenue_data) if all_revenue_data else pd.DataFrame()

# 3. revenue_df가 이미 존재한다고 가정하고 TTM 추가
print("📊 TTM 매출 컬럼 추가 중...")

if 'revenue_df' in globals() and not revenue_df.empty:
    # TTM 컬럼 추가
    revenue_df_with_ttm = add_revenue_ttm(revenue_df)

    # TTM을 십억 단위로 변환
    revenue_df_with_ttm['revenue_ttm_billions'] = revenue_df_with_ttm['revenue_ttm'] / 1_000_000_000

    # 월말 날짜 컬럼 추가
    print("📅 revenue_df_with_ttm에 월말 날짜 컬럼 추가 중...")
    revenue_df_with_ttm['date_month_end'] = revenue_df_with_ttm['date'].apply(convert_to_month_end)

    print(f"✅ revenue_df_with_ttm 생성 완료: {revenue_df_with_ttm.shape}")

    # TTM 데이터 샘플 확인
    print(f"\n📋 TTM 매출 데이터 샘플 (최신 5개):")
    sample_cols = ['ticker', 'date', 'date_month_end', 'period', 'revenue_billions', 'revenue_ttm_billions']
    available_cols = [col for col in sample_cols if col in revenue_df_with_ttm.columns]
    print(revenue_df_with_ttm[available_cols].tail(5).to_string(index=False))

else:
    print("❌ revenue_df가 존재하지 않습니다. 먼저 revenue_df를 생성해주세요.")

    # revenue_df가 없는 경우를 위한 샘플 생성 코드
    print("\n revenue_df 생성이 필요한 경우, 다음과 같은 구조여야 합니다:")
    print("필수 컬럼: ['ticker', 'date', 'period', 'revenue', 'revenue_billions', ...]")

# 4. revenue_df_with_ttm 확인 함수
def check_revenue_ttm():
    """revenue_df_with_ttm 존재 및 구조 확인"""
    if 'revenue_df_with_ttm' in globals():
        df = revenue_df_with_ttm
        print(f"✅ revenue_df_with_ttm 존재: {df.shape}")
        print(f"   컬럼: {list(df.columns)}")
        print(f"   날짜 범위: {df['date'].min()} ~ {df['date'].max()}")

        if 'revenue_ttm' in df.columns:
            print(f"   TTM 데이터: 평균 ${df['revenue_ttm'].mean()/1e9:.2f}B")
        if 'date_month_end' in df.columns:
            print(f"   월말 날짜: 변환 완료")
    else:
        print("❌ revenue_df_with_ttm이 존재하지 않습니다.")


if not revenue_df.empty:
    # 날짜 컬럼 처리
    revenue_df['date'] = pd.to_datetime(revenue_df['date'])
    revenue_df = revenue_df.sort_values(['ticker', 'date'], ascending=[True, True])

    # 월말 날짜 컬럼 추가
    print("📅 revenue_df에 월말 날짜 컬럼 추가 중...")
    revenue_df['date_month_end'] = revenue_df['date'].apply(convert_to_month_end)

    print(f"✅ revenue_df 생성 완료: {len(revenue_df)} 레코드")

    # TTM 컬럼 추가
    print("📊 TTM 매출 컬럼 추가 중...")
    revenue_df_with_ttm = add_revenue_ttm(revenue_df)
    revenue_df_with_ttm['revenue_ttm_billions'] = revenue_df_with_ttm['revenue_ttm'] / 1_000_000_000

    # TTM DataFrame에도 월말 날짜 컬럼 추가
    print("📅 revenue_df_with_ttm에 월말 날짜 컬럼 추가 중...")
    revenue_df_with_ttm['date_month_end'] = revenue_df_with_ttm['date'].apply(convert_to_month_end)

    print(f"✅ TTM 컬럼 추가 완료")

    # TTM 데이터 샘플 확인
    print(f"\n📋 TTM 매출 데이터 샘플 (최신 5개):")
    ttm_sample = revenue_df_with_ttm[['ticker', 'date', 'date_month_end', 'period', 'revenue_billions', 'revenue_ttm_billions']].tail(5)
    print(ttm_sample.to_string(index=False))

✅ API 키 설정됨: hT0gAk87j9...
🔍 API 연결 테스트 중...
API 연결 성공
🚀 경량 데이터 수집 시작
📊 대상 기업: 1개 - ['SMCI']
📈 수집 데이터: 분기별 매출 + 월별 시가총액
📈 Fetching quarterly revenue data...


Revenue:   0%|          | 0/1 [00:00<?, ?it/s]

   ✅ SMCI: 84개 분기


Revenue: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]

📊 TTM 매출 컬럼 추가 중...
📅 revenue_df_with_ttm에 월말 날짜 컬럼 추가 중...
✅ revenue_df_with_ttm 생성 완료: (84, 11)

📋 TTM 매출 데이터 샘플 (최신 5개):
ticker       date date_month_end period  revenue_billions  revenue_ttm_billions
  SMCI 2024-06-30     2024-06-30     Q4              5.35             14.989251
  SMCI 2024-09-30     2024-09-30     Q1              5.94             18.806835
  SMCI 2024-12-31     2024-12-31     Q2              5.68             20.819873
  SMCI 2025-03-31     2025-03-31     Q3              4.60             21.569720
  SMCI 2025-06-30     2025-06-30     Q4              5.76             21.972042
📅 revenue_df에 월말 날짜 컬럼 추가 중...
✅ revenue_df 생성 완료: 84 레코드
📊 TTM 매출 컬럼 추가 중...
📅 revenue_df_with_ttm에 월말 날짜 컬럼 추가 중...
✅ TTM 컬럼 추가 완료

📋 TTM 매출 데이터 샘플 (최신 5개):
ticker       date date_month_end period  revenue_billions  revenue_ttm_billions
  SMCI 2024-06-30     2024-06-30     Q4              5.35             14.989251
  SMCI 2024-09-30     2024-09-30     Q1              5.94             18.8068

In [9]:
def fetch_market_data_yearly(ticker, start_year=2010):
    """연도별로 세분화해서 데이터 수집"""
    all_data = []
    current_year = datetime.now().year

    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"

        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': API_KEY}

        print(f"{year}년 데이터 수집 중...")

        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
                    print(f"  {year}년: {len(data)}개 데이터")
                else:
                    print(f"  {year}년: 데이터 없음")
            else:
                print(f"  {year}년: HTTP {response.status_code}")

            time.sleep(0.3)  # API 제한 고려

        except Exception as e:
            print(f"  {year}년 오류: {str(e)}")

    print(f"총 수집 데이터: {len(all_data)}개")
    return all_data if all_data else None, None


def process_daily_to_monthly_market_data(daily_data):
    """일별 시가총액 데이터를 월말 기준으로 변환"""
    if not daily_data:
        print("데이터가 없습니다.")
        return pd.DataFrame()

    print(f"일별 데이터 처리 시작: {len(daily_data)}개 레코드")

    # DataFrame 생성
    df = pd.DataFrame(daily_data)

    # 날짜 컬럼 처리
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')

    # 연월 컬럼 추가 (그룹화용)
    df['year_month'] = df['date'].dt.to_period('M')

    print(f"날짜 범위: {df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
    print(f"총 월수: {df['year_month'].nunique()}개월")

    # 각 월의 마지막 날짜 데이터만 추출
    monthly_data = []

    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]

        # 해당 월의 가장 마지막 날짜 데이터 선택
        last_day_data = month_data.loc[month_data['date'].idxmax()]

        monthly_data.append({
            'ticker': last_day_data.get('symbol', 'UNKNOWN'),  # API에서는 'symbol'로 올 수 있음
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
            'year_month': year_month
        })

    # DataFrame 생성
    monthly_df = pd.DataFrame(monthly_data)

    print(f"월별 데이터 생성 완료: {len(monthly_df)}개 레코드")

    return monthly_df

# 연도별 데이터 수집 및 월별 변환 통합 함수
def fetch_and_process_yearly_market_data(ticker, start_year=2010):
    """연도별 수집 후 월별로 변환"""
    print(f"=== {ticker} 연도별 데이터 수집 및 월별 변환 ===")

    # 연도별 데이터 수집
    data, error = fetch_market_data_yearly(ticker, start_year)

    if not data:
        print(f"데이터 수집 실패: {error}")
        return pd.DataFrame()

    # 월별 데이터로 변환
    monthly_df = process_daily_to_monthly_market_data(data)

    if monthly_df.empty:
        print("월별 변환 실패")
        return pd.DataFrame()

    # convert_to_month_end 적용
    print("월말 날짜로 표준화 중...")
    monthly_df['date_month_end'] = monthly_df['date'].apply(convert_to_month_end)

    # 불필요한 컬럼 제거
    marketcap_df = monthly_df[['ticker', 'date', 'date_month_end', 'market_cap', 'market_cap_billions']].copy()

    print(f"최종 marketcap_df 생성 완료: {marketcap_df.shape}")
    print(f"날짜 범위: {marketcap_df['date_month_end'].min().strftime('%Y-%m-%d')} ~ {marketcap_df['date_month_end'].max().strftime('%Y-%m-%d')}")

    # 샘플 데이터 출력
    print("\n월말 시가총액 데이터 샘플 (최신 5개):")
    print(marketcap_df.tail(5)[['ticker', 'date', 'date_month_end', 'market_cap_billions']].to_string(index=False))

    return marketcap_df

# 실행 코드
print("AAPL 시가총액 데이터 수집 및 월별 변환 시작...")

# 연도별 데이터 수집 후 월별 변환
marketcap_df = fetch_and_process_yearly_market_data(ticker, 2010)

AAPL 시가총액 데이터 수집 및 월별 변환 시작...
=== SMCI 연도별 데이터 수집 및 월별 변환 ===
2010년 데이터 수집 중...
  2010년: 252개 데이터
2011년 데이터 수집 중...
  2011년: 252개 데이터
2012년 데이터 수집 중...
  2012년: 250개 데이터
2013년 데이터 수집 중...
  2013년: 252개 데이터
2014년 데이터 수집 중...
  2014년: 252개 데이터
2015년 데이터 수집 중...
  2015년: 252개 데이터
2016년 데이터 수집 중...
  2016년: 252개 데이터
2017년 데이터 수집 중...
  2017년: 251개 데이터
2018년 데이터 수집 중...
  2018년: 251개 데이터
2019년 데이터 수집 중...
  2019년: 252개 데이터
2020년 데이터 수집 중...
  2020년: 253개 데이터
2021년 데이터 수집 중...
  2021년: 252개 데이터
2022년 데이터 수집 중...
  2022년: 251개 데이터
2023년 데이터 수집 중...
  2023년: 250개 데이터
2024년 데이터 수집 중...
  2024년: 252개 데이터
2025년 데이터 수집 중...
  2025년: 177개 데이터
총 수집 데이터: 3951개
일별 데이터 처리 시작: 3951개 레코드
날짜 범위: 2010-01-04 ~ 2025-09-17
총 월수: 189개월
월별 데이터 생성 완료: 189개 레코드
월말 날짜로 표준화 중...
최종 marketcap_df 생성 완료: (189, 5)
날짜 범위: 2010-01-31 ~ 2025-09-30

월말 시가총액 데이터 샘플 (최신 5개):
ticker       date date_month_end  market_cap_billions
  SMCI 2025-05-30     2025-05-31                23.92
  SMCI 2025-06-30     2025-06-30           

#### 4. PSR 데이터 측정

In [10]:
# revenue_df_with_ttm에서 필요한 컬럼 추출
ttm_data = revenue_df_with_ttm[['ticker', 'date_month_end', 'revenue_billions', 'revenue_ttm_billions']].copy()

# monthly_df에서 필요한 컬럼 추출
market_data = marketcap_df[['ticker', 'date_month_end', 'market_cap_billions']].copy()

# 데이터 병합 (left join)
merged_data = pd.merge(
    market_data,
    ttm_data,
    on=['ticker', 'date_month_end'],
    how='left'
)

# 종목별, 날짜별 정렬
merged_data = merged_data.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

print("   2. Forward fill 적용 (limit=3)...")

# 종목별로 그룹화하여 forward fill 적용
merged_data['revenue_ttm_billions'] = merged_data.groupby('ticker')['revenue_ttm_billions'].ffill(limit=3)
merged_data['revenue_billions'] = merged_data['revenue_billions'].ffill(limit=3)
print("   3. 결측치 제거...")

# 병합 전 레코드 수
records_before_dropna = len(merged_data)

# 결측치 제거
merged_data = merged_data.dropna(subset=['revenue_ttm_billions']).reset_index(drop=True)


# ==============================================
# TTM Shift 및 월별 PSR 계산
# ==============================================

print("📊 TTM 데이터 shift 및 PSR 계산 중...")

# revenue_ttm_billions를 2개월 뒤로 shift (종목별로)
merged_data['revenue_ttm_shift'] = merged_data.groupby('ticker')['revenue_ttm_billions'].shift(2)

print("   ✅ revenue_ttm_billions를 2개월 뒤로 shift 완료")

# 월별 PSR_ttm 계산 (시가총액 / TTM 매출)
merged_data['PSR_ttm'] = merged_data['market_cap_billions'] / merged_data['revenue_ttm_shift']

print("   ✅ PSR_ttm 계산 완료 (market_cap_billions / revenue_ttm_shift)")

# shift로 인한 NaN 값 제거
records_before_nan_removal = len(merged_data)
merged_data = merged_data.dropna(subset=['revenue_ttm_shift', 'PSR_ttm']).reset_index(drop=True)
records_after_nan_removal = len(merged_data)

print(f"   ✅ NaN 값 제거 완료")
print(f"     - NaN 제거 전: {records_before_nan_removal:,}개 레코드")
print(f"     - NaN 제거 후: {records_after_nan_removal:,}개 레코드")
print(f"     - 제거된 레코드: {records_before_nan_removal - records_after_nan_removal:,}개")

   2. Forward fill 적용 (limit=3)...
   3. 결측치 제거...
📊 TTM 데이터 shift 및 PSR 계산 중...
   ✅ revenue_ttm_billions를 2개월 뒤로 shift 완료
   ✅ PSR_ttm 계산 완료 (market_cap_billions / revenue_ttm_shift)
   ✅ NaN 값 제거 완료
     - NaN 제거 전: 187개 레코드
     - NaN 제거 후: 185개 레코드
     - 제거된 레코드: 2개


#### 5. 외생변수 입력

In [11]:
from sqlalchemy import create_engine

def get_hs_data(hs_code_6d, db_info):
    """
    HS Code로 무역 데이터 추출

    Parameters:
    - hs_code_6d (str): 6자리 HS Code
    - db_info (dict): 데이터베이스 연결 정보

    Returns:
    - pd.DataFrame: 추출된 데이터
    """

    try:
        # 데이터베이스 연결
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        # 데이터 조회
        query = f"""
        SELECT * FROM us_trade_monthly_data_with_forecast
        WHERE hs_code_6d = '{hs_code_6d}'
        ORDER BY date DESC
        """

        df = pd.read_sql(query, engine)
        engine.dispose()

        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])

        return df

    except Exception as e:
        print(f"오류: {str(e)}")
        return pd.DataFrame()

def get_latest_input_date_data(df):
    """
    DataFrame에서 input_date가 가장 최근인 데이터만 추출

    Parameters:
    - df (pd.DataFrame): 원본 데이터프레임

    Returns:
    - pd.DataFrame: 가장 최근 input_date의 데이터
    """

    # input_date 컬럼이 존재하는지 확인
    if 'input_date' not in df.columns:
        print("Error: 'input_date' 컬럼이 존재하지 않습니다.")
        return pd.DataFrame()

    # input_date를 datetime으로 변환 (이미 datetime이면 그대로)
    df_copy = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df_copy['input_date']):
        df_copy['input_date'] = pd.to_datetime(df_copy['input_date'])

    # 가장 최근 input_date 찾기
    latest_date = df_copy['input_date'].max()

    # 가장 최근 날짜의 데이터만 필터링
    latest_data = df_copy[df_copy['input_date'] == latest_date].copy()

    print(f"가장 최근 input_date: {latest_date.strftime('%Y-%m-%d')}")
    print(f"해당 날짜의 데이터: {len(latest_data):,}개")

    return latest_data

In [12]:
root_hs_code = '854232'   # 로그 변환 적용 HS Code
FORECAST_STEPS = 15                # 예측 개월 수
MIN_PERIODS = 60                   # 최소 데이터 개수(5년)

# DB 접속 정보
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': '3307',
    'database': 'investar'
}

# HS Code로 데이터 추출
export_df = get_hs_data('851762', db_info)

# 사용 예시 (export_df가 있다고 가정)
if 'export_df' in globals():
    # 가장 최근 input_date 데이터 추출
    latest_export_data = get_latest_input_date_data(export_df)
    latest_export_data = latest_export_data.sort_values('date').reset_index(drop=True)
    # 결과 확인
    if not latest_export_data.empty:
        print("\n추출된 데이터 샘플:")
        display_cols = ['hs_code_6d', 'date', 'export_value', 'input_date']
        available_cols = [col for col in display_cols if col in latest_export_data.columns]
        print(latest_export_data[available_cols].head().to_string(index=False))
else:
    print("export_df 변수가 존재하지 않습니다.")


가장 최근 input_date: 2025-09-02
해당 날짜의 데이터: 165개

추출된 데이터 샘플:
hs_code_6d       date input_date
    851762 2013-01-31 2025-09-02
    851762 2013-02-28 2025-09-02
    851762 2013-03-31 2025-09-02
    851762 2013-04-30 2025-09-02
    851762 2013-05-31 2025-09-02


#### 4. 최종 데이터의 결합

In [14]:
print("데이터프레임 결합 시작...")

# 1. latest_export_data에서 필요한 컬럼 추출
export_subset = latest_export_data[['hs_code_6d', 'date', 'expDlr']].copy()

print(f"수출 데이터: {len(export_subset)}개 레코드")

# 2. latest_export_data의 date를 월말로 변환
export_subset['date'] = pd.to_datetime(export_subset['date'])
export_subset['date_month_end'] = export_subset['date'].apply(convert_to_month_end)

print("수출 데이터 월말 변환 완료")

# 3. merged_data와 결합 (left join - merged_data 기준)
final_data = pd.merge(
    export_subset[['date_month_end', 'hs_code_6d', 'expDlr']],  # 필요한 컬럼만
    merged_data,
    on='date_month_end',
    how='left'
)

print(f"데이터 결합 완료!")
print(f"- merged_data: {len(merged_data)} 레코드")
print(f"- export_subset: {len(export_subset)} 레코드")
print(f"- final_data: {len(final_data)} 레코드")

# 4. 결과 확인
if not final_data.empty:
    print(f"\n결합된 데이터 샘플 (상위 5개):")
    display_cols = ['ticker', 'date_month_end', 'market_cap_billions', 'revenue_ttm_shift', 'PSR_ttm', 'hs_code_6d', 'expDlr']
    available_cols = [col for col in display_cols if col in final_data.columns]
    print(final_data[available_cols].head().to_string(index=False))

    # 결측치 확인
    print(f"\n결측치 현황:")
    null_counts = final_data.isnull().sum()
    for col in ['hs_code_6d', 'expDlr']:
        if col in null_counts.index:
            print(f"- {col}: {null_counts[col]}개")

    # expDlr이 있는 데이터 개수
    if 'expDlr' in final_data.columns:
        valid_export = final_data['expDlr'].notna().sum()
        print(f"- 수출 데이터 매칭: {valid_export}개/{len(final_data)}개")

else:
    print("결합 실패")

print(f"\nfinal_data 변수로 접근 가능합니다.")

데이터프레임 결합 시작...
수출 데이터: 165개 레코드
수출 데이터 월말 변환 완료
데이터 결합 완료!
- merged_data: 185 레코드
- export_subset: 165 레코드
- final_data: 165 레코드

결합된 데이터 샘플 (상위 5개):
ticker date_month_end  market_cap_billions  revenue_ttm_shift  PSR_ttm hs_code_6d       expDlr
  SMCI     2013-01-31                 0.52           1.036696 0.501594     851762 1233740000.0
  SMCI     2013-02-28                 0.49           1.078268 0.454432     851762 1178980000.0
  SMCI     2013-03-31                 0.48           1.078268 0.445158     851762 1378080000.0
  SMCI     2013-04-30                 0.41           1.078268 0.380239     851762 1284210000.0
  SMCI     2013-05-31                 0.44           1.116124 0.394221     851762 1275680000.0

결측치 현황:
- hs_code_6d: 0개
- expDlr: 0개
- 수출 데이터 매칭: 165개/165개

final_data 변수로 접근 가능합니다.


#### 5. SARIMA 매출예측

In [16]:
# 외생변수를 포함한 SARIMA 예측 실행 코드 (완전 수정 버전)
# 위의 함수들이 먼저 실행되어 있어야 함

print("=== 외생변수를 포함한 SARIMA 예측 시작 (사용 가능한 기간까지) ===")

# 설정 변수
USE_EXOGENOUS = True  # True: 외생변수 사용, False: 외생변수 미사용

print(f"외생변수 사용 설정: {USE_EXOGENOUS}")

# 1. 예측 가능한 기간 먼저 확인
print("\n1. 예측 가능한 기간 확인")
available_periods, last_quarterly_date, available_forecast_dates = calculate_available_forecast_periods(
    quarterly_data, final_data
)

if available_periods == 0:
    print("외생변수 데이터가 부족하여 예측할 수 없습니다.")
    sarima_exog_final_result = None
    sarima_exog_final_revenue = None
    sarima_exog_final_model_type = None
else:
    print(f"\n예측 가능한 분기: {available_periods}개")
    print(f"예측 날짜: {[d.strftime('%Y-Q%m') for d in available_forecast_dates]}")

    # 2. 외생변수 사용 모드로 예측
    print("\n2. 외생변수 사용 SARIMA 예측")
    sarima_exog_results = sarima_forecast_comparison(
        quarterly_data,
        final_data,
        USE_EXOGENOUS=True
    )

    if sarima_exog_results['forecast_values'] is not None:
        print("외생변수 SARIMA 예측 성공!")

        # 실제 사용된 예측 날짜 (사용 가능한 날짜)
        actual_forecast_dates = sarima_exog_results['available_dates']

        # 결과 DataFrame 생성
        sarima_exog_forecast_result = create_sarima_exog_result_dataframe(
            actual_forecast_dates,
            sarima_exog_results['forecast_values'],
            sarima_exog_results['param_string'],
            use_exogenous=True
        )

        # 변수 정리
        sarima_exog_historical_revenue = quarterly_data['revenue_billions'].tolist()
        sarima_exog_forecast_revenue = sarima_exog_results['forecast_values'].tolist()
        sarima_exog_model_info = sarima_exog_results['model_info']

        print("\n외생변수 SARIMA 예측 결과:")
        print(sarima_exog_forecast_result)

        print(f"\n모델 정보:")
        print(f"  - 파라미터: {sarima_exog_results['param_string']}")
        print(f"  - AIC: {sarima_exog_model_info['aic']:.2f}")
        print(f"  - 외생변수 사용: {sarima_exog_model_info['exog_used']}")
        print(f"  - 실제 예측 스텝: {sarima_exog_model_info['actual_forecast_steps']}")
        if sarima_exog_model_info.get('future_exog'):
            print(f"  - 사용된 외생변수 값: {[f'{x:.2f}%' for x in sarima_exog_model_info['future_exog']]}")

        print(f"\n예측값:")
        for i, (date, value) in enumerate(zip(actual_forecast_dates, sarima_exog_forecast_revenue)):
            print(f"  {date.strftime('%Y-Q%m')}: {value:.2f} billions")

        # 3. USE_EXOGENOUS 설정에 따른 최종 결과
        print(f"\n3. USE_EXOGENOUS = {USE_EXOGENOUS} 설정에 따른 최종 결과")

        if USE_EXOGENOUS:
            final_sarima_forecast_result = sarima_exog_forecast_result
            final_sarima_forecast_revenue = sarima_exog_forecast_revenue
            final_sarima_model_type = "SARIMA_with_exogenous"
            print("최종 결과: 외생변수를 포함한 SARIMA 예측 사용")
        else:
            final_sarima_forecast_result = None
            final_sarima_forecast_revenue = None
            final_sarima_model_type = "SARIMA_without_exogenous"
            print("최종 결과: 외생변수 미사용으로 None 처리")

        # 최종 변수들을 전역으로 설정
        sarima_exog_final_result = final_sarima_forecast_result
        sarima_exog_final_revenue = final_sarima_forecast_revenue
        sarima_exog_final_model_type = final_sarima_model_type

    else:
        print("외생변수 SARIMA 예측 실패")
        sarima_exog_final_result = None
        sarima_exog_final_revenue = None
        sarima_exog_final_model_type = None

# 4. 외생변수 미사용 모드 테스트
print(f"\n4. 외생변수 미사용 모드 테스트")
sarima_no_exog_results = sarima_forecast_comparison(
    quarterly_data,
    final_data,
    USE_EXOGENOUS=False
)
print("외생변수 미사용 - None 처리됨")

# 5. 결과 요약
print(f"\n=== SARIMA 외생변수 예측 결과 요약 ===")
print(f"USE_EXOGENOUS 설정: {USE_EXOGENOUS}")

if USE_EXOGENOUS and 'sarima_exog_final_result' in locals() and sarima_exog_final_result is not None:
    print(f"외생변수 사용 예측 성공:")
    print(f"  - 모델: {sarima_exog_final_model_type}")
    print(f"  - 예측 가능 분기: {available_periods}개")
    print(f"  - 예측값 범위: {min(sarima_exog_final_revenue):.2f} ~ {max(sarima_exog_final_revenue):.2f} billions")
    print(f"  - 예측 기간: {available_forecast_dates[0].strftime('%Y-Q%m')} ~ {available_forecast_dates[-1].strftime('%Y-Q%m')}")
else:
    print("외생변수 미사용 또는 예측 실패로 None 처리")
    sarima_exog_final_result = None
    sarima_exog_final_revenue = None
    sarima_exog_final_model_type = None

# 6. 데이터 확인 (디버깅용) - 단순화
print(f"\n=== 데이터 확인 ===")
if 'sarima_exog_results' in locals() and sarima_exog_results.get('prepared_data') is not None:
    prepared_data = sarima_exog_results['prepared_data']
    print(f"준비된 데이터:")
    print(f"  - 기간: {prepared_data['date'].min()} ~ {prepared_data['date'].max()}")
    print(f"  - 데이터 길이: {len(prepared_data)}")
    print(f"  - 컬럼들: {list(prepared_data.columns)}")
    if 'endog_var' in prepared_data.columns:
        print(f"  - 매출 범위: {prepared_data['endog_var'].min():.2f} ~ {prepared_data['endog_var'].max():.2f}")
    if 'exog_var' in prepared_data.columns:
        print(f"  - 외생변수 범위: {prepared_data['exog_var'].min():.2f}% ~ {prepared_data['exog_var'].max():.2f}%")
        print(f"  - 외생변수 샘플: {prepared_data['exog_var'].tail(3).tolist()}")
else:
    print("준비된 데이터가 없습니다.")

print("\n=== 외생변수 SARIMA 예측 완료 ===")

# 7. 사용 예시 (설정 변경)
print(f"\n=== 설정 변경 예시 ===")
print("# 외생변수 사용하려면:")
print("USE_EXOGENOUS = True")
print("\n# 외생변수 사용하지 않으려면:")
print("USE_EXOGENOUS = False")
print("\n# 그 후 위의 코드를 다시 실행하면 됩니다.")

# 8. 간단한 개별 테스트 (필요시)
print(f"\n=== 간단한 개별 테스트 ===")
print("개별 함수 호출 예시:")
print("# 1. 예측 가능 기간 확인")
print("available_periods, _, dates = calculate_available_forecast_periods(quarterly_data, final_data)")
print("# 2. 개별 예측 실행")
# 외생변수를 포함한 SARIMA 예측 실행 코드 (완전 수정 버전)
# 위의 함수들이 먼저 실행되어 있어야 함

print("=== 외생변수를 포함한 SARIMA 예측 시작 (사용 가능한 기간까지) ===")

# 설정 변수
USE_EXOGENOUS = True  # True: 외생변수 사용, False: 외생변수 미사용

print(f"외생변수 사용 설정: {USE_EXOGENOUS}")

# 1. 예측 가능한 기간 먼저 확인
print("\n1. 예측 가능한 기간 확인")
available_periods, last_quarterly_date, available_forecast_dates = calculate_available_forecast_periods(
    quarterly_data, final_data
)

if available_periods == 0:
    print("외생변수 데이터가 부족하여 예측할 수 없습니다.")
    sarima_exog_final_result = None
    sarima_exog_final_revenue = None
    sarima_exog_final_model_type = None
else:
    print(f"\n예측 가능한 분기: {available_periods}개")
    print(f"예측 날짜: {[d.strftime('%Y-Q%m') for d in available_forecast_dates]}")

    # 2. 외생변수 사용 모드로 예측
    print("\n2. 외생변수 사용 SARIMA 예측")
    sarima_exog_results = sarima_forecast_comparison(
        quarterly_data,
        final_data,
        USE_EXOGENOUS=True
    )

    if sarima_exog_results['forecast_values'] is not None:
        print("외생변수 SARIMA 예측 성공!")

        # 실제 사용된 예측 날짜 (사용 가능한 날짜)
        actual_forecast_dates = sarima_exog_results['available_dates']

        # 결과 DataFrame 생성
        sarima_exog_forecast_result = create_sarima_exog_result_dataframe(
            actual_forecast_dates,
            sarima_exog_results['forecast_values'],
            sarima_exog_results['param_string'],
            use_exogenous=True
        )

        # 변수 정리
        sarima_exog_historical_revenue = quarterly_data['revenue_billions'].tolist()
        sarima_exog_forecast_revenue = sarima_exog_results['forecast_values'].tolist()
        sarima_exog_model_info = sarima_exog_results['model_info']

        print("\n외생변수 SARIMA 예측 결과:")
        print(sarima_exog_forecast_result)

        print(f"\n모델 정보:")
        print(f"  - 파라미터: {sarima_exog_results['param_string']}")
        print(f"  - AIC: {sarima_exog_model_info['aic']:.2f}")
        print(f"  - 외생변수 사용: {sarima_exog_model_info['exog_used']}")
        print(f"  - 실제 예측 스텝: {sarima_exog_model_info['actual_forecast_steps']}")
        if sarima_exog_model_info.get('future_exog'):
            print(f"  - 사용된 외생변수 값: {[f'{x:.2f}%' for x in sarima_exog_model_info['future_exog']]}")

        print(f"\n예측값:")
        for i, (date, value) in enumerate(zip(actual_forecast_dates, sarima_exog_forecast_revenue)):
            print(f"  {date.strftime('%Y-Q%m')}: {value:.2f} billions")

        # 3. USE_EXOGENOUS 설정에 따른 최종 결과
        print(f"\n3. USE_EXOGENOUS = {USE_EXOGENOUS} 설정에 따른 최종 결과")

        if USE_EXOGENOUS:
            final_sarima_forecast_result = sarima_exog_forecast_result
            final_sarima_forecast_revenue = sarima_exog_forecast_revenue
            final_sarima_model_type = "SARIMA_with_exogenous"
            print("최종 결과: 외생변수를 포함한 SARIMA 예측 사용")
        else:
            final_sarima_forecast_result = None
            final_sarima_forecast_revenue = None
            final_sarima_model_type = "SARIMA_without_exogenous"
            print("최종 결과: 외생변수 미사용으로 None 처리")

        # 최종 변수들을 전역으로 설정
        sarima_exog_final_result = final_sarima_forecast_result
        sarima_exog_final_revenue = final_sarima_forecast_revenue
        sarima_exog_final_model_type = final_sarima_model_type

    else:
        print("외생변수 SARIMA 예측 실패")
        sarima_exog_final_result = None
        sarima_exog_final_revenue = None
        sarima_exog_final_model_type = None

# 4. 외생변수 미사용 모드 테스트
print(f"\n4. 외생변수 미사용 모드 테스트")
sarima_no_exog_results = sarima_forecast_comparison(
    quarterly_data,
    final_data,
    USE_EXOGENOUS=False
)
print("외생변수 미사용 - None 처리됨")

# 5. 결과 요약
print(f"\n=== SARIMA 외생변수 예측 결과 요약 ===")
print(f"USE_EXOGENOUS 설정: {USE_EXOGENOUS}")

if USE_EXOGENOUS and 'sarima_exog_final_result' in locals() and sarima_exog_final_result is not None:
    print(f"외생변수 사용 예측 성공:")
    print(f"  - 모델: {sarima_exog_final_model_type}")
    print(f"  - 예측 가능 분기: {available_periods}개")
    print(f"  - 예측값 범위: {min(sarima_exog_final_revenue):.2f} ~ {max(sarima_exog_final_revenue):.2f} billions")
    print(f"  - 예측 기간: {available_forecast_dates[0].strftime('%Y-Q%m')} ~ {available_forecast_dates[-1].strftime('%Y-Q%m')}")
else:
    print("외생변수 미사용 또는 예측 실패로 None 처리")
    sarima_exog_final_result = None
    sarima_exog_final_revenue = None
    sarima_exog_final_model_type = None

# 6. 데이터 확인 (디버깅용) - 단순화
print(f"\n=== 데이터 확인 ===")
if 'sarima_exog_results' in locals() and sarima_exog_results.get('prepared_data') is not None:
    prepared_data = sarima_exog_results['prepared_data']
    print(f"준비된 데이터:")
    print(f"  - 기간: {prepared_data['date'].min()} ~ {prepared_data['date'].max()}")
    print(f"  - 데이터 길이: {len(prepared_data)}")
    print(f"  - 컬럼들: {list(prepared_data.columns)}")
    if 'endog_var' in prepared_data.columns:
        print(f"  - 매출 범위: {prepared_data['endog_var'].min():.2f} ~ {prepared_data['endog_var'].max():.2f}")
    if 'exog_var' in prepared_data.columns:
        print(f"  - 외생변수 범위: {prepared_data['exog_var'].min():.2f}% ~ {prepared_data['exog_var'].max():.2f}%")
        print(f"  - 외생변수 샘플: {prepared_data['exog_var'].tail(3).tolist()}")
else:
    print("준비된 데이터가 없습니다.")

print("\n=== 외생변수 SARIMA 예측 완료 ===")

# 7. 사용 예시 (설정 변경)
print(f"\n=== 설정 변경 예시 ===")
print("# 외생변수 사용하려면:")
print("USE_EXOGENOUS = True")
print("\n# 외생변수 사용하지 않으려면:")
print("USE_EXOGENOUS = False")
print("\n# 그 후 위의 코드를 다시 실행하면 됩니다.")

# 8. 간단한 개별 테스트 (필요시)
print(f"\n=== 간단한 개별 테스트 ===")
print("개별 함수 호출 예시:")
print("# 1. 예측 가능 기간 확인")
print("available_periods, _, dates = calculate_available_forecast_periods(quarterly_data, final_data)")
print("# 2. 개별 예측 실행")


=== 외생변수를 포함한 SARIMA 예측 시작 (사용 가능한 기간까지) ===
외생변수 사용 설정: True

1. 예측 가능한 기간 확인


NameError: name 'quarterly_data' is not defined

In [146]:
# final_data에서 예측 구간 확인
print("=== final_data 예측 구간 확인 ===")

# 1. expDlr과 revenue_billions 상태 확인
print("expDlr 상태:")
print(f"  - 전체 길이: {len(final_data)}")
print(f"  - expDlr NaN 개수: {final_data['expDlr'].isna().sum()}")
print(f"  - expDlr 마지막 10개 값:")
print(final_data[['date_month_end', 'expDlr']].tail(10))

print("\nrevenue_billions 상태:")
print(f"  - revenue_billions NaN 개수: {final_data['revenue_billions'].isna().sum()}")
print(f"  - revenue_billions 마지막 10개 값:")
print(final_data[['date_month_end', 'revenue_billions']].tail(10))

# 2. 예측 구간 찾기
forecast_mask = (
    final_data['expDlr'].notna() &
    final_data['revenue_billions'].isna()
)

forecast_data = final_data[forecast_mask]
print(f"\n예측 구간 데이터:")
print(f"  - 예측 구간 길이: {len(forecast_data)}")
if len(forecast_data) > 0:
    print(f"  - 예측 구간 날짜: {forecast_data['date_month_end'].min()} ~ {forecast_data['date_month_end'].max()}")
    print("  - 예측 구간 상세:")
    print(forecast_data[['date_month_end', 'expDlr', 'revenue_billions']].head())
else:
    print("  - 예측 구간이 없습니다!")

    # 대안: expDlr이 있고 분기별 데이터 이후인 구간 찾기
    last_quarterly_date = pd.to_datetime(quarterly_data['date_month_end'].iloc[-1])
    future_mask = (
        final_data['expDlr'].notna() &
        (pd.to_datetime(final_data['date_month_end']) > last_quarterly_date)
    )
    future_data = final_data[future_mask]
    print(f"\n  - 분기 데이터 이후 expDlr 있는 구간: {len(future_data)}개")
    if len(future_data) > 0:
        print(f"  - 해당 구간 날짜: {future_data['date_month_end'].min()} ~ {future_data['date_month_end'].max()}")

=== final_data 예측 구간 확인 ===
expDlr 상태:
  - 전체 길이: 165
  - expDlr NaN 개수: 0
  - expDlr 마지막 10개 값:
    date_month_end        expDlr
155     2025-12-31  2.207550e+09
156     2026-01-31  2.260790e+09
157     2026-02-28  2.110310e+09
158     2026-03-31  2.455920e+09
159     2026-04-30  2.260900e+09
160     2026-05-31  2.206470e+09
161     2026-06-30  2.299530e+09
162     2026-07-31  2.385440e+09
163     2026-08-31  2.394650e+09
164     2026-09-30  2.239720e+09

revenue_billions 상태:
  - revenue_billions NaN 개수: 12
  - revenue_billions 마지막 10개 값:
    date_month_end  revenue_billions
155     2025-12-31               NaN
156     2026-01-31               NaN
157     2026-02-28               NaN
158     2026-03-31               NaN
159     2026-04-30               NaN
160     2026-05-31               NaN
161     2026-06-30               NaN
162     2026-07-31               NaN
163     2026-08-31               NaN
164     2026-09-30               NaN

예측 구간 데이터:
  - 예측 구간 길이: 12
  - 예측 구간 날짜: 2025

In [147]:
 sarima_exog_final_result

In [148]:
# LSTM 예측 실행
revenue_series = quarterly_data['revenue_billions']
lstm_forecast_values, lstm_model, lstm_scaler = lstm_forecast(revenue_series, lookback_window=8, forecast_steps=4)

# 예측 날짜 생성 (SARIMA와 동일한 방식)
last_date = pd.to_datetime(quarterly_data['date_month_end'].iloc[-1])
lstm_forecast_dates = []
for i in range(1, 5):
    next_quarter_date = last_date + pd.DateOffset(months=3*i)
    quarter_end = pd.Timestamp(
        year=next_quarter_date.year,
        month=next_quarter_date.month,
        day=pd.Timestamp(next_quarter_date.year, next_quarter_date.month, 1).days_in_month
    )
    lstm_forecast_dates.append(quarter_end)

# LSTM 예측 결과 변수들
lstm_forecast_result = pd.DataFrame({
    'date': lstm_forecast_dates,
    'forecast_revenue_billions': lstm_forecast_values,
    'year': [d.year for d in lstm_forecast_dates],
    'quarter': [d.quarter for d in lstm_forecast_dates],
    'year_quarter': [f"{d.year}Q{d.quarter}" for d in lstm_forecast_dates]
})

lstm_historical_revenue = revenue_series.tolist()
lstm_forecast_revenue = lstm_forecast_values.tolist()

# 모델 성능 평가 (옵션)
lstm_performance = evaluate_model_performance(revenue_series, lstm_model, lstm_scaler)

NaN values filled using interpolation
Data range: 0.28 to 5.94
Data length: 55
Training data shape: X=(47, 8, 1), y=(47,)
Training completed. Final loss: 0.003174
Forecast completed: [5.499368  5.4789042 5.481003  5.485342 ]
NaN found in actual data, filling with interpolation
Evaluation: train_size=44, test_size=11
Performance - MSE: 0.0913, MAE: 0.2544, RMSE: 0.3021


In [149]:
# 개별 함수들을 바로 호출해서 사용 가능
revenue_series = quarterly_data['revenue_billions']
date_series = quarterly_data['date_month_end']

# 데이터 준비
prophet_data = prepare_prophet_data(date_series, revenue_series)

# 예측 수행
forecast_result, prophet_model, future_data, full_forecast = prophet_forecast(
    prophet_data,
    forecast_periods=4
)

# 결과 정리
prophet_forecast_result = create_forecast_result_dataframe(forecast_result)

# 성능 평가
prophet_performance = evaluate_prophet_performance(prophet_data, prophet_model)

DEBUG:cmdstanpy:cmd: where.exe tbb.dll
cwd: None
DEBUG:cmdstanpy:TBB already found in load path


NaN values filled using forward/backward fill
Prophet data prepared: 55 records from 2013-03-31 00:00:00 to 2026-09-30 00:00:00


DEBUG:cmdstanpy:input tempfile: C:\Users\82108\AppData\Local\Temp\tmpz3nwad92\n68fot5p.json
DEBUG:cmdstanpy:input tempfile: C:\Users\82108\AppData\Local\Temp\tmpz3nwad92\v2up68l6.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['C:\\Users\\82108\\AppData\\Local\\Programs\\Python\\Python39\\Lib\\site-packages\\prophet\\stan_model\\prophet_model.bin', 'random', 'seed=92312', 'data', 'file=C:\\Users\\82108\\AppData\\Local\\Temp\\tmpz3nwad92\\n68fot5p.json', 'init=C:\\Users\\82108\\AppData\\Local\\Temp\\tmpz3nwad92\\v2up68l6.json', 'output', 'file=C:\\Users\\82108\\AppData\\Local\\Temp\\tmpz3nwad92\\prophet_modelwnps_tfw\\prophet_model-20250916031357.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
03:13:57 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing


Fitting Prophet model...


03:13:57 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
DEBUG:cmdstanpy:cmd: where.exe tbb.dll
cwd: None
DEBUG:cmdstanpy:TBB already found in load path


Making predictions for 4 periods...
Forecast completed. Values range: 4.53 to 5.16
Performance evaluation: train_size=51, test_size=4
Performance evaluation failed: 'name'


In [55]:
# 수정된 Exponential Smoothing 예측 실행 코드
# 위의 수정된 함수들이 먼저 실행되어 있어야 함

print("=== 수정된 Exponential Smoothing 시계열 예측 시작 ===")

# 1. 데이터 품질 확인
print("\n1. 데이터 품질 확인")
revenue_series = quarterly_data['revenue_billions']
data_quality = check_data_quality(revenue_series, "Revenue Data")

# 2. 수정된 함수로 Damped vs Non-Damped 비교 실행
print("\n2. 수정된 Damped vs Non-Damped 모델 비교")
comparison_results_fixed = compare_damped_vs_non_damped_fixed(
    revenue_series,
    forecast_steps=4,
    seasonal='add',  # 가법 계절성
    seasonal_periods=4,  # 분기별 계절성
    trend='add'  # 가법 트렌드
)

# 3. Non-Damped 결과 처리
if comparison_results_fixed['non_damped']['forecast'] is not None:
    print("\n=== Non-Damped 예측 결과 ===")

    # 날짜 생성
    last_date = pd.to_datetime(quarterly_data['date_month_end'].iloc[-1])
    es_non_damped_forecast_dates = generate_forecast_dates(last_date, periods=4, freq='Q')

    # 결과 DataFrame 생성
    es_non_damped_forecast_result = create_exponential_smoothing_result_dataframe(
        es_non_damped_forecast_dates,
        comparison_results_fixed['non_damped']['forecast'],
        forecast_lower=comparison_results_fixed['non_damped']['details']['forecast_lower'] if comparison_results_fixed['non_damped']['details'] else None,
        forecast_upper=comparison_results_fixed['non_damped']['details']['forecast_upper'] if comparison_results_fixed['non_damped']['details'] else None,
        damped=False
    )

    # 변수 정리
    es_non_damped_historical_revenue = revenue_series.tolist()
    es_non_damped_forecast_revenue = comparison_results_fixed['non_damped']['forecast'].tolist()
    es_non_damped_model = comparison_results_fixed['non_damped']['model']
    es_non_damped_performance = comparison_results_fixed['non_damped']['performance']

    print("Non-Damped 예측 결과:")
    print(es_non_damped_forecast_result)

    if 'Error' not in es_non_damped_performance:
        print(f"\nNon-Damped 성능:")
        print(f"  - MSE: {es_non_damped_performance['MSE']:.4f}")
        print(f"  - MAE: {es_non_damped_performance['MAE']:.4f}")
        print(f"  - RMSE: {es_non_damped_performance['RMSE']:.4f}")
        print(f"  - MAPE: {es_non_damped_performance['MAPE']:.2f}%")
else:
    print("Non-Damped 예측 실패")

# 4. Damped 결과 처리
if comparison_results_fixed['damped']['forecast'] is not None:
    print("\n=== Damped 예측 결과 ===")

    # 날짜 생성 (동일한 날짜)
    last_date = pd.to_datetime(quarterly_data['date_month_end'].iloc[-1])
    es_damped_forecast_dates = generate_forecast_dates(last_date, periods=4, freq='Q')

    # 결과 DataFrame 생성
    es_damped_forecast_result = create_exponential_smoothing_result_dataframe(
        es_damped_forecast_dates,
        comparison_results_fixed['damped']['forecast'],
        forecast_lower=comparison_results_fixed['damped']['details']['forecast_lower'] if comparison_results_fixed['damped']['details'] else None,
        forecast_upper=comparison_results_fixed['damped']['details']['forecast_upper'] if comparison_results_fixed['damped']['details'] else None,
        damped=True
    )

    # 변수 정리
    es_damped_historical_revenue = revenue_series.tolist()
    es_damped_forecast_revenue = comparison_results_fixed['damped']['forecast'].tolist()
    es_damped_model = comparison_results_fixed['damped']['model']
    es_damped_performance = comparison_results_fixed['damped']['performance']

    print("Damped 예측 결과:")
    print(es_damped_forecast_result)

    if 'Error' not in es_damped_performance:
        print(f"\nDamped 성능:")
        print(f"  - MSE: {es_damped_performance['MSE']:.4f}")
        print(f"  - MAE: {es_damped_performance['MAE']:.4f}")
        print(f"  - RMSE: {es_damped_performance['RMSE']:.4f}")
        print(f"  - MAPE: {es_damped_performance['MAPE']:.2f}%")
else:
    print("Damped 예측 실패")

# 5. 최적 모델 선택 및 결과
print("\n=== 최종 비교 결과 ===")

if 'best_model' in comparison_results_fixed:
    best_model_type = comparison_results_fixed['best_model']
    print(f"최적 모델: {best_model_type}")

    if best_model_type == 'non_damped':
        print("Non-Damped 모델이 더 좋은 성능을 보입니다.")
        if 'es_non_damped_forecast_result' in locals():
            best_forecast_result = es_non_damped_forecast_result
            best_forecast_values = es_non_damped_forecast_revenue
            best_performance = es_non_damped_performance
    else:
        print("Damped 모델이 더 좋은 성능을 보입니다.")
        if 'es_damped_forecast_result' in locals():
            best_forecast_result = es_damped_forecast_result
            best_forecast_values = es_damped_forecast_revenue
            best_performance = es_damped_performance

    # 최적 모델 결과를 기본 변수로 설정
    if 'best_forecast_result' in locals():
        es_forecast_result = best_forecast_result
        es_forecast_revenue = best_forecast_values
        es_performance = best_performance
        es_best_model_type = best_model_type

        print(f"\n최적 모델 예측값:")
        for i, (date, value) in enumerate(zip(best_forecast_result['date'], best_forecast_values)):
            print(f"  {date.strftime('%Y-Q%m')}: {value:.2f} billions")

# 6. 간단한 개별 모델 테스트
print("\n=== 간단한 개별 모델 테스트 ===")

# 간단한 Non-Damped 테스트
print("\n개별 Non-Damped 예측 (간단 버전):")
try:
    clean_data = prepare_exponential_smoothing_data(revenue_series)
    simple_model = ExponentialSmoothing(clean_data, trend='add', damped_trend=False)
    simple_fitted = simple_model.fit()
    simple_forecast = simple_fitted.forecast(steps=4)
    print(f"Simple Non-Damped 예측: {simple_forecast.values}")
except Exception as e:
    print(f"Simple Non-Damped 실패: {e}")

# 간단한 Damped 테스트
print("\n개별 Damped 예측 (간단 버전):")
try:
    clean_data = prepare_exponential_smoothing_data(revenue_series)
    simple_damped_model = ExponentialSmoothing(clean_data, trend='add', damped_trend=True)
    simple_damped_fitted = simple_damped_model.fit()
    simple_damped_forecast = simple_damped_fitted.forecast(steps=4)
    print(f"Simple Damped 예측: {simple_damped_forecast.values}")
except Exception as e:
    print(f"Simple Damped 실패: {e}")

print("\n=== 수정된 Exponential Smoothing 예측 완료 ===")

=== 수정된 Exponential Smoothing 시계열 예측 시작 ===

1. 데이터 품질 확인

=== Revenue Data Quality Check ===
Length: 55
NaN count: 4
Infinite count: 0
Min value: 0.2800
Max value: 5.9400
Mean value: 1.5214
Data type: float64

2. 수정된 Damped vs Non-Damped 모델 비교
=== Damped vs Non-Damped 비교 ===

1. Non-Damped 예측:
=== Exponential Smoothing 예측 시작 (Damped: False) ===
NaN values filled using forward/backward fill
Data prepared: 55 records, range: 0.28 to 5.94
Model configuration:
  - Trend: add
  - Seasonal: add
  - Seasonal periods: 4
  - Damped: False
  - Auto optimize: True
Fitting Exponential Smoothing model...
Model fitting completed
Fitted parameters:
  - smoothing_level: 1.0000
  - smoothing_trend: 0.0000
  - smoothing_seasonal: 0.0000
  - damping_trend: nan
  - initial_level: 0.3314
  - initial_trend: 0.0983
  - initial_seasons: [-0.1498651492361596, 0.08539928441535935, 0.020627089065554993]
  - use_boxcox: 0.0000
  - lamda: None
  - remove_bias: 0.0000
Generating 4 forecasts...
신뢰구간 계산 실패: 'HoltWin

In [56]:
es_forecast_revenue

[5.811270544251393, 5.594420167102976, 5.840534062433961, 5.784010658741843]

In [57]:
simple_damped_forecast

55    5.847542
56    5.929661
57    6.011369
58    6.092668
dtype: float64

In [58]:
final_data

,date_month_end,hs_code_6d,expDlr,ticker,market_cap_billions,revenue_billions,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm
0,2013-01-31,851762,1.233740e+09,SMCI,0.52,0.29,1.078268,1.036696,0.501594
1,2013-02-28,851762,1.178980e+09,SMCI,0.49,0.29,1.078268,1.078268,0.454432
2,2013-03-31,851762,1.378080e+09,SMCI,0.48,0.28,1.116124,1.078268,0.445158
3,2013-04-30,851762,1.284210e+09,SMCI,0.41,0.28,1.116124,1.078268,0.380239
4,2013-05-31,851762,1.275680e+09,SMCI,0.44,0.28,1.116124,1.116124,0.394221
...,...,...,...,...,...,...,...,...,...
160,2026-05-31,851762,2.206470e+09,NaN,NaN,NaN,NaN,NaN,NaN
161,2026-06-30,851762,2.299530e+09,NaN,NaN,NaN,NaN,NaN,NaN
162,2026-07-31,851762,2.385440e+09,NaN,NaN,NaN,NaN,NaN,NaN
163,2026-08-31,851762,2.394650e+09,NaN,NaN,NaN,NaN,NaN,NaN
